<a href="https://colab.research.google.com/github/Tecknique/200_ml/blob/main/FLA_UW_GUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# /content/uwlfa_gui_cells/cell1_config.py

import io
import json
import re
import shutil
import threading
import zipfile
import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional

from IPython.display import HTML, display

# ----------------- config -----------------
OWNER, REPO = "timrobinson", "UW-LFA-Analysis"
BRANCH = "main"
BASE_REL = Path("100microliters/Database")
IMG_EXTS = {".tif", ".tiff", ".jpg", ".jpeg", ".png"}

EXPORT_ROOT = Path("/content/roi_exports") if Path("/content").exists() else Path.cwd() / "roi_exports"
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

# ✅ RECOMMENDED: keep each run's exports in its own dated folder
# This matches how your UI selects dates (via /exports) and how cell8 prints the save path.
DATE_FOLDER = datetime.datetime.now().strftime("%Y-%m-%d")


In [ ]:
# /content/uwlfa_gui_cells/cell2_deps.py

from pathlib import Path

# IMPORTANT:
# - Do NOT override EXPORT_ROOT / DATE_FOLDER here.
# - Do NOT delete export folders automatically here.
# Cell1 should define EXPORT_ROOT and DATE_FOLDER (or you can set them there).

try:
    import cv2
    import numpy as np
    import requests
    import pandas as pd
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from scipy.signal import savgol_filter

except Exception:
    import sys
    import subprocess as sp

    sp.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "opencv-python-headless",
            "requests",
            "flask",
            "matplotlib",
            "pandas",
            "scipy",
        ],
        check=True,
    )

    import cv2  # noqa
    import numpy as np  # noqa
    import requests  # noqa
    import pandas as pd  # noqa
    import matplotlib  # noqa

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt  # noqa
    from scipy.signal import savgol_filter  # noqa


# ----------------- safety checks -----------------
# If cell1 didn't define these for some reason, define safe defaults.
if "EXPORT_ROOT" not in globals():
    EXPORT_ROOT = Path("/content/roi_exports") if Path("/content").exists() else Path.cwd() / "roi_exports"
    EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

if "DATE_FOLDER" not in globals():
    # Leave empty/neutral; better than accidentally deleting/overwriting older dates.
    DATE_FOLDER = ""

print("Deps loaded.")
print(f"EXPORT_ROOT = {EXPORT_ROOT}")
print(f"DATE_FOLDER = {DATE_FOLDER!r}")


# ----------------- OPTIONAL: manual delete (OFF by default) -----------------
# If you REALLY want to delete a specific date folder, flip this to True
# and set DELETE_DATE_FOLDER to that folder name. Then run the cell once.
ENABLE_MANUAL_DELETE = False
DELETE_DATE_FOLDER = ""  # e.g. "2-6-2026" or "2026-02-06" exactly as it exists on disk

if ENABLE_MANUAL_DELETE:
    import shutil

    if not DELETE_DATE_FOLDER:
        raise ValueError("ENABLE_MANUAL_DELETE=True but DELETE_DATE_FOLDER is empty.")

    folder_to_delete = Path(EXPORT_ROOT) / DELETE_DATE_FOLDER
    if folder_to_delete.exists() and folder_to_delete.is_dir():
        print(f"Deleting folder: {folder_to_delete.as_posix()}")
        shutil.rmtree(folder_to_delete)
        print("Folder deleted successfully.")
    else:
        print(f"Folder not found: {folder_to_delete.as_posix()}")


Deps loaded.
EXPORT_ROOT = /content/roi_exports
DATE_FOLDER = ''


In [ ]:
# /content/uwlfa_gui_cells/cell3_helpers.py

def _download_repo_zip(owner: str, repo: str, branch: str) -> bytes:
    def fetch(br: str) -> Optional[bytes]:
        url = f"https://github.com/{owner}/{repo}/archive/refs/heads/{br}.zip"
        r = requests.get(url, timeout=60)
        return r.content if r.status_code == 200 else None

    blob = fetch(branch) or (fetch("master") if branch == "main" else None)
    if blob is None:
        raise RuntimeError(f"Cannot download {owner}/{repo} ({branch}/master).")
    return blob


def _extract_zip_to_tmp(zip_bytes: bytes, base_dir: Path) -> Path:
    tmp_root = base_dir / "_uwlfa_tmp"
    if tmp_root.exists():
        shutil.rmtree(tmp_root)
    tmp_root.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(io.BytesIO(zip_bytes), "r") as zf:
        zf.extractall(tmp_root)
        top_dirs = sorted({Path(n).parts[0] for n in zf.namelist() if "/" in n})

    if not top_dirs:
        raise RuntimeError("Unexpected zip structure.")
    return tmp_root / top_dirs[0]


def sanitize_name(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", (s or "").strip())


def row_name_from_filename(filename: str) -> str:
    stem = Path(filename).stem
    if "__" in stem:
        return stem.split("__", 1)[0]
    return stem.split("_", 1)[0] if "_" in stem else stem


def row_sort_key(row: str) -> Tuple[int, float, str]:
    r = (row or "").strip()
    loads_order = {
        "neg": (0, -1.0),
        "1e5": (1, 1e5),
        "1.5e5": (2, 1.5e5),
        "5e5": (3, 5e5),
        "1e6": (4, 1e6),
        "5e6": (5, 5e6),
        "1e7": (6, 1e7),
        "cc": (7, 7e7),
        "k": (8, 8e7),
        "dip": (9, 9e7),
        "pos": (10, 1e12),
    }
    key = loads_order.get(r.lower())
    if key:
        return (0, key[0], r.lower())
    m = re.match(r"^(\d+(?:\.\d+)?)e(\d+)$", r.lower())
    if m:
        base = float(m.group(1))
        exp = float(m.group(2))
        return (1, base * (10**exp), r.lower())
    return (2, float("inf"), r.lower())


def _png_message(msg: str) -> bytes:
    fig, ax = plt.subplots(figsize=(7, 2.2))
    ax.text(0.02, 0.6, msg, ha="left", va="center", fontsize=10)
    ax.set_axis_off()
    buf = io.BytesIO()
    fig.tight_layout()
    fig.savefig(buf, format="png", dpi=150)
    plt.close(fig)
    buf.seek(0)
    return buf.getvalue()


def _read_image_meta(path: str) -> Tuple[int, int, int]:
    try:
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if img is None:
            return 0, 0, 0
        h, w = img.shape[:2]
        c = 1 if img.ndim == 2 else int(img.shape[2])
        return int(w), int(h), int(c)
    except Exception:
        return 0, 0, 0


In [ ]:
# /content/uwlfa_gui_cells/cell4_load_repo.py

base_dir = Path("/content") if Path("/content").exists() else Path.cwd()
zip_bytes = _download_repo_zip(OWNER, REPO, BRANCH)
REPO_DIR = _extract_zip_to_tmp(zip_bytes, base_dir)
DB_DIR = REPO_DIR / BASE_REL
assert DB_DIR.exists(), f"Missing path: {DB_DIR}"

DATASETS: Dict[str, List[Dict]] = {}

for sub in sorted([p for p in DB_DIR.iterdir() if p.is_dir()], key=lambda p: p.name.lower()):
    files_out: List[Dict] = []
    for p in sorted(sub.rglob("*")):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            w, h, c = _read_image_meta(p.as_posix())
            fname = p.name
            files_out.append(
                {
                    "filename": fname,
                    "path": p.as_posix(),
                    "width": w,
                    "height": h,
                    "channels": c,
                    "row": row_name_from_filename(fname),
                }
            )
    DATASETS[sub.name] = files_out

payload = {
    "repo_dir": str(REPO_DIR),
    "db_dir": str(DB_DIR),
    "folders": [{"name": k, "files": v} for k, v in DATASETS.items()],
}


In [ ]:
# /content/uwlfa_gui_cells/cell5_flask_template_part1.py

from flask import Flask, jsonify, request, Response  # noqa

# app must exist BEFORE cell7 uses @app.route decorators
app = Flask(__name__)



TEMPLATE_HTML_1 = r"""
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1" />
<title>UWLFA GUI</title>
<style>
  :root{
    --bg:#0b1220;
    --panel:#0f1a2b;
    --panel2:#101f36;
    --text:#e6edf3;
    --muted:rgba(230,237,243,0.68);
    --border:rgba(230,237,243,0.14);
    --border2:rgba(230,237,243,0.10);
    --btn:#1f3b64;
    --btn2:#233e67;
    --good:rgba(34,197,94,0.16);
    --bad:rgba(239,68,68,0.16);
    --mono: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Liberation Mono","Courier New", monospace;
    --sans: system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,sans-serif;
  }
  html,body{height:100%;}
  body{
    margin:0;
    background:linear-gradient(180deg,#080f1d 0%, #070c16 100%);
    color:var(--text);
    font-family:var(--sans);
  }
  .wrap{
    max-width: 1280px;
    margin: 18px auto;
    padding: 0 14px 40px;
  }
  .topbar{
    display:flex;
    align-items:center;
    justify-content:space-between;
    margin-bottom:12px;
  }
  .brand{
    font-weight:700;
    font-size:18px;
    letter-spacing:0.2px;
  }
  .muted{ color: var(--muted); }
  .mono{ font-family: var(--mono); }
  .grid{
    display:grid;
    grid-template-columns: 1.2fr 1fr;
    gap: 14px;
  }
  @media (max-width: 980px){
    .grid{ grid-template-columns: 1fr; }
  }
  .card{
    background: rgba(16, 31, 54, 0.72);
    border:1px solid var(--border);
    border-radius: 16px;
    box-shadow: 0 10px 30px rgba(0,0,0,0.30);
    overflow:hidden;
  }
  .card .hd{
    padding: 12px 14px;
    border-bottom: 1px solid var(--border2);
    display:flex;
    align-items:center;
    justify-content:space-between;
    gap: 10px;
  }
  .card .hd .title{
    font-size: 14px;
    font-weight: 700;
    letter-spacing: 0.2px;
  }
  .card .bd{
    padding: 12px 14px;
  }
  .row{
    display:flex;
    align-items:center;
    gap: 10px;
    flex-wrap:wrap;
    margin-bottom: 10px;
  }
  select, input[type="text"], input[type="number"]{
    background: rgba(0,0,0,0.25);
    color: var(--text);
    border:1px solid var(--border);
    border-radius: 10px;
    padding: 8px 10px;
    outline:none;
  }
  input[type="number"]{ width: 90px; }
  select{ max-width: 340px; }
  button{
    background: linear-gradient(180deg, var(--btn2), var(--btn));
    color: var(--text);
    border:1px solid var(--border);
    border-radius: 12px;
    padding: 8px 12px;
    cursor:pointer;
    font-weight: 600;
    letter-spacing: 0.15px;
  }
  button:hover{ filter: brightness(1.05); }
  button:active{ transform: translateY(1px); }
  .btn-green{
    background: linear-gradient(180deg, rgba(34,197,94,0.9), rgba(34,197,94,0.55));
  }
  .btn-gray{
    background: linear-gradient(180deg, rgba(148,163,184,0.35), rgba(148,163,184,0.20));
  }
  .btn-red{
    background: linear-gradient(180deg, rgba(239,68,68,0.65), rgba(239,68,68,0.35));
  }
  .split{
    display:grid;
    grid-template-columns: 1fr 1fr;
    gap: 12px;
  }
  @media (max-width: 700px){
    .split{ grid-template-columns: 1fr; }
  }

  /* Image viewer */
  .imgWrap{
    position: relative;
    display:flex;
    justify-content:center;
    align-items:center;
    background: rgba(0,0,0,0.22);
    border: 1px solid var(--border2);
    border-radius: 14px;
    padding: 10px;
    min-height: 260px;
  }
  #img{
    max-width: 100%;
    max-height: 520px;
    border-radius: 12px;
    border: 1px solid var(--border2);
  }
  #img.loading{
    opacity: 0.65;
    filter: blur(0.2px);
  }
  #rect{
    position:absolute;
    display:none;
    border: 2px dashed rgba(59,130,246,0.9);
    background: rgba(59,130,246,0.12);
    pointer-events:none;
    border-radius: 10px;
  }

  /* SBR plot canvas */
  #sbrCanvas{
    width: 100%;
    height: 300px;
    background: rgba(0,0,0,0.18);
    border: 1px solid var(--border2);
    border-radius: 14px;
    display:none;
  }
  .kv{
    display:grid;
    grid-template-columns: 1fr 1fr;
    gap: 10px 14px;
  }
  .k{
    display:flex;
    justify-content:space-between;
    gap: 12px;
    padding: 8px 10px;
    background: rgba(0,0,0,0.18);
    border: 1px solid var(--border2);
    border-radius: 12px;
    align-items:center;
  }

  /* Tables */
  table{
    width: 100%;
    border-collapse: collapse;
    font-size: 13px;
  }
  th, td{
    border: 1px solid var(--border2);
    padding: 8px 10px;
    vertical-align: top;
  }
  th{
    text-align:left;
    background: rgba(0,0,0,0.22);
  }
  .cellOK{ background: var(--good); }
  .cellBad{ background: var(--bad); }
  details summary{
    cursor:pointer;
    user-select:none;
    margin-top: 6px;
  }
  pre{
    white-space: pre-wrap;
    word-break: break-word;
    background: rgba(0,0,0,0.22);
    border: 1px solid var(--border2);
    border-radius: 12px;
    padding: 10px;
    overflow:auto;
  }

  /* Log */
  #log{
    width: 100%;
    height: 190px;
    background: rgba(0,0,0,0.25);
    border: 1px solid var(--border2);
    border-radius: 14px;
    padding: 10px;
    overflow:auto;
    box-sizing:border-box;
    font-family: var(--mono);
    font-size: 12px;
    line-height: 1.35;
  }
  .pill{
    display:inline-block;
    padding: 2px 8px;
    border-radius: 999px;
    border: 1px solid var(--border2);
    background: rgba(0,0,0,0.18);
    font-size: 12px;
    color: var(--muted);
  }
  .hr{
    height:1px;
    background: var(--border2);
    margin: 12px 0;
  }
</style>
</head>

<body>
<div class="wrap">
  <div class="topbar">
    <div class="brand">UWLFA GUI <span class="pill">ROI · SBR · Export</span></div>
    <div class="muted mono" style="font-size:12px">Template payload injected via __PAYLOAD__</div>
  </div>

  <div class="grid">

    <!-- LEFT COLUMN -->
    <div class="card">
      <div class="hd">
        <div class="title">Image Browser + ROI Saver</div>
        <div class="muted mono" style="font-size:12px">
          Row: <input id="rowName" type="text" value="" style="width:140px" readonly />
        </div>
      </div>

      <div class="bd">
        <div class="row">
          <label class="muted">Dataset</label>
          <select id="ds"></select>
          <button id="load">Load Dataset</button>

          <label class="muted">Image</label>
          <select id="imgsel"></select>
        </div>

        <div class="row">
          <button id="select">Select ROI</button>
          <button id="reset" class="btn-gray">Reset View</button>
          <button id="rotate" class="btn-gray">Rotate 90°</button>
          <button id="applyGray" class="btn-gray">Apply Grayscale</button>
          <button id="save" class="btn-green">Save ROI</button>
          <button id="export" class="btn-green">Export CSV for Dataset</button>
        </div>

        <div class="imgWrap">
          <img id="img" alt="image" />
          <div id="rect"></div>
        </div>

        <div class="hr"></div>

        <div class="row">
          <label class="muted">Export path</label>
          <input id="savePath" type="text" value="" style="flex:1;min-width:240px" readonly />
          <button id="copyPath" class="btn-gray">Copy</button>
        </div>

        <div class="hr"></div>

        <div class="title muted" style="font-weight:700;margin-bottom:8px">Log</div>
        <pre id="log"></pre>
      </div>
    </div>

    <!-- RIGHT COLUMN -->
    <div class="card">
      <div class="hd">
        <div class="title">ROI Grid + SBR + Demo + Final Graph</div>
        <div class="muted mono" style="font-size:12px">Interactive analysis panel</div>
      </div>

      <div class="bd">

        <!-- ROI Grid -->
        <div class="title" style="font-weight:800;font-size:13px;margin-bottom:8px">ROI Grid (date + dataset family)</div>

        <div class="row">
          <label class="muted">Date</label>
          <select id="dfDate"></select>
          <button id="refreshDfDates" class="btn-gray">Refresh Dates</button>

          <label class="muted">Dataset</label>
          <select id="gridDataset"></select>

          <label class="muted">Include dip</label>
          <select id="gridDip">
            <option value="0" selected>0</option>
            <option value="1">1</option>
          </select>

          <button id="plotGrid" class="btn-green">Plot ROI Grid</button>
        </div>

        <img id="dfGrid" style="display:none;width:100%;border-radius:14px;border:1px solid rgba(230,237,243,0.10);" />

        <div class="hr"></div>

        <!-- SBR Interactive -->
        <div class="title" style="font-weight:800;font-size:13px;margin-bottom:8px">SBR Interactive (choose row + replicate)</div>

        <div class="row">
          <label class="muted">Row</label>
          <select id="sbrRow"></select>

          <label class="muted">Replicate</label>
          <select id="sbrRep">
            <option value="N1" selected>N1</option>
            <option value="N2">N2</option>
            <option value="N3">N3</option>
          </select>

          <label class="muted">Index</label>
          <input id="sbrIndex" type="number" value="0" min="0" />

          <button id="plotSBR" class="btn-green">Plot SBR</button>
          <button id="resetLines" class="btn-gray">Reset Lines</button>
        </div>

        <div id="sbrMsg" class="muted mono" style="font-size:12px;margin-bottom:8px"></div>

        <canvas id="sbrCanvas"></canvas>

        <div class="kv" style="margin-top:10px">
          <div class="k"><span class="muted">Blue</span><span id="blueInfo" class="mono">-</span></div>
          <div class="k"><span class="muted">Red</span><span id="redInfo" class="mono">-</span></div>
          <div class="k"><span class="muted">Peak</span><span id="greenInfo" class="mono">-</span></div>
          <div class="k"><span class="muted">Baseline@Peak</span><span id="yellowInfo" class="mono">-</span></div>
        </div>

        <div class="hr"></div>

        <!-- SBR Table -->
        <div class="row" style="justify-content:space-between">
          <div class="title" style="font-weight:800;font-size:13px">SBR Table (rows × N1/N2/N3)</div>
          <div>
            <button id="loadSbrTable" class="btn-gray">Load Table</button>
          </div>
        </div>
        <div id="sbrTablePanel" style="display:none">
          <div id="sbrTableMeta" class="muted mono" style="font-size:12px;margin:8px 0"></div>
          <div id="sbrTableBody"></div>
        </div>

        <div class="hr"></div>

        <!-- Positivity threshold -->
        <div class="row" style="justify-content:space-between">
          <div class="title" style="font-weight:800;font-size:13px">Positivity Threshold (neg only)</div>
          <div>
            <button id="computePT" class="btn-gray">Compute PT</button>
          </div>
        </div>

        <div id="ptPanel" style="display:none">
          <div class="kv" style="margin-top:10px">
            <div class="k"><span class="muted">μ</span><span id="ptMu" class="mono">-</span></div>
            <div class="k"><span class="muted">σ</span><span id="ptSigma" class="mono">-</span></div>
            <div class="k"><span class="muted">Threshold</span><span id="ptThr" class="mono">-</span></div>
            <div class="k"><span class="muted">Row</span><span class="mono">neg</span></div>
          </div>
          <div id="ptBody" style="margin-top:10px"></div>
        </div>

        <div class="hr"></div>

        <!-- Demo View -->
        <div class="title" style="font-weight:800;font-size:13px;margin-bottom:8px">
          Demo view (matches notebook plots + full results dict)
        </div>

        <div class="row">
          <button id="runDemo" class="btn-gray">Run Demo (plots)</button>
          <button id="loadDemoJson" class="btn-gray">Load Raw Demo Data (JSON)</button>
          <button id="addExp" class="btn-green">Add as Experiment</button>
          <button id="clearExp" class="btn-gray">Clear Experiments</button>
        </div>

        <div id="demoMsg" class="muted mono" style="font-size:12px;margin-bottom:8px"></div>

        <div class="split">
          <div class="card" style="border-radius:14px">
            <div class="hd"><div class="title">Row mean±SEM</div></div>
            <div class="bd">
              <div class="mono" id="demoRowStats">-</div>
              <img id="demoStrip" style="display:none;width:100%;margin-top:10px;border-radius:12px;border:1px solid rgba(230,237,243,0.10);" />
              <img id="demoAverage" style="display:none;width:100%;margin-top:10px;border-radius:12px;border:1px solid rgba(230,237,243,0.10);" />
            </div>
          </div>

          <div class="card" style="border-radius:14px">
            <div class="hd"><div class="title">Pooled mean±SEM</div></div>
            <div class="bd">
              <div class="mono" id="demoPooledStats">-</div>
              <img id="expPlot" style="display:none;width:100%;margin-top:10px;border-radius:12px;border:1px solid rgba(230,237,243,0.10);" />
            </div>
          </div>
        </div>

        <div style="margin-top:10px">
          <div class="muted" style="font-weight:700;margin:8px 0">Raw demo JSON</div>
          <pre id="demoJson" style="max-height:240px"></pre>
        </div>

        <div class="hr"></div>

        <!-- Final graph -->
        <div class="title" style="font-weight:800;font-size:13px;margin-bottom:8px">Final Graph</div>

        <div class="row">
          <label class="muted">Mode</label>
          <select id="finalMode">
            <option value="combined" selected>combined</option>
            <option value="N1">N1</option>
            <option value="N2">N2</option>
            <option value="N3">N3</option>
            <option value="legacy">legacy</option>
          </select>

          <label class="muted">Legacy source</label>
          <select id="finalSource">
            <option value="df" selected>df</option>
            <option value="disk">disk</option>
          </select>

          <button id="plotFinal" class="btn-green">Plot Final</button>
          <span id="finalMsg" class="muted mono" style="font-size:12px"></span>
        </div>

        <img id="finalGraph" style="display:none;width:100%;border-radius:14px;border:1px solid rgba(230,237,243,0.10);" />

      </div>
    </div>

  </div>
</div>

<!-- __PAYLOAD__ injected by Flask home() -->
<script>
  window.PAYLOAD = __PAYLOAD__;
</script>
"""


In [ ]:
TEMPLATE_HTML_2 = r"""
<script>
const LOG = (m) => {
  const el = document.getElementById('log');
  el.textContent += m + "\n";
  el.scrollTop = el.scrollHeight;
};
const MSG = (m)=>{document.getElementById('sbrMsg').textContent = m || "";};

/* =========================================================
   Helpers: "active export context" (ROI Grid Date + Dataset)
   ========================================================= */
function getExportDate(){
  return (typeof DFDATES !== 'undefined' && DFDATES && DFDATES.value) ? DFDATES.value : '';
}
function getExportDatasetFamily(){
  const el = document.getElementById('gridDataset');
  return el ? (el.value || "") : (window.DF_DATASET || "");
}
function getSbrReplicate(){
  const el = document.getElementById('sbrRep');
  return el ? (el.value || "N1") : "N1";
}

async function loadDfFor(dataset, date){
  if(!dataset) return {ok:false, msg:"missing dataset"};
  const payload = { dataset: dataset };
  if(date && date !== '(none)') payload.date = date;

  try{
    const r = await fetch('/load_df', {
      method:'POST',
      headers:{'Content-Type':'application/json'},
      body: JSON.stringify(payload)
    });
    const j = await r.json();
    if(!j.ok){
      LOG(j.msg || 'Load DF: not available yet.');
      return j;
    }
    window.DF_DATASET = dataset;
    LOG(j.msg || `Loaded DF for ${dataset}`);
    return j;
  }catch(e){
    LOG('Load DF failed: ' + e);
    return {ok:false, msg:String(e)};
  }
}

async function fillSavePath(){
  try{
    const r = await fetch('/export_info'); const j = await r.json();
    document.getElementById('savePath').value = j.full;
  }catch(e){}
}
document.getElementById('copyPath').onclick = ()=>{
  const el = document.getElementById('savePath'); el.select(); document.execCommand('copy');
};

const DS = document.getElementById('ds');
const IMGSEL = document.getElementById('imgsel');
const IMG = document.getElementById('img');
const RECT = document.getElementById('rect');
const ROWNAME = document.getElementById('rowName');
const GRID_DS = document.getElementById('gridDataset');

let mouseDown=false, startX=0, startY=0;
let currentDataset = null, currentIndex = null;
let imgBusy = false;
let roiMode=false;

/* =========================
   NEW: SBR Table (rows × N1/N2/N3)
   ========================= */
const sbrTablePanel = document.getElementById('sbrTablePanel');
const sbrTableBody  = document.getElementById('sbrTableBody');
const sbrTableMeta  = document.getElementById('sbrTableMeta');

function fmt6(x){
  const v = Number(x);
  if(!Number.isFinite(v)) return "nan";
  return v.toFixed(6);
}

function renderCellDetails(cell){
  if(!cell || !cell.ok){
    return `<div class="muted">${cell?.msg || "missing"}</div>`;
  }

  // supports both "saved" and "recomputed" modes + UI/original fields
  const signal = Number(
    cell.peak_y ??
    cell.signal_value ??
    cell.signal ??
    cell.peak_y ??
    cell.peak_value
  );

  const base = Number(
    cell.baseline_at_peak ??
    cell.baseline_value ??
    cell.baseline ??
    cell.baseline_at_peak ??
    cell.baseline_value
  );

  const sbr = Number(cell.sbr);

  const eq = cell.equations || {};
  const lines = [
    eq.sbr_symbolic || eq.symbolic || "SBR = signal / baseline",
    eq.sbr_substitute || eq.substitute || "",
    eq.sbr_result || eq.result || ""
  ].filter(x => String(x).trim().length);

  const win = cell.windows ? JSON.stringify(cell.windows, null, 2) : "";

  return `
    <div class="mono" style="white-space:pre-wrap;margin:6px 0 10px">
${lines.join("\n")}
    </div>
    <div class="muted">
      signal=<span class="mono">${fmt6(signal)}</span>,
      baseline=<span class="mono">${fmt6(base)}</span>,
      sbr=<span class="mono">${fmt6(sbr)}</span>
    </div>
    ${win ? `<div class="muted" style="margin-top:10px">windows:</div><pre class="mono" style="max-height:220px">${win}</pre>` : ``}
  `;
}

function renderSbrTable(table){
  const rows = table.rows || [];
  const reps = table.replicates || ["N1","N2","N3"];
  const cells = table.cells || {};

  let html = `<table><thead><tr><th>Row</th>`;
  for(const rep of reps){
    html += `<th>${rep}</th>`;
  }
  html += `</tr></thead><tbody>`;

  for(const row of rows){
    html += `<tr><td class="mono">${row}</td>`;
    for(const rep of reps){
      const cell = (cells[row] && cells[row][rep]) ? cells[row][rep] : null;
      const ok = !!(cell && cell.ok);
      const sbr = ok ? Number(cell.sbr) : NaN;

      html += `<td class="${ok ? 'cellOK' : 'cellBad'}">`;
      if(ok){
        html += `
          <div class="mono" style="font-size:13px"><b>${fmt6(sbr)}</b></div>
          <details style="margin-top:8px">
            <summary>Math</summary>
            ${renderCellDetails(cell)}
          </details>
        `;
      } else {
        html += `<div class="muted">${cell?.msg || "missing"}</div>`;
      }
      html += `</td>`;
    }
    html += `</tr>`;
  }

  html += `</tbody></table>`;
  return html;
}

async function loadSbrTable(){
  try{
    const date = getExportDate();
    const ds = getExportDatasetFamily();
    if(!date || date === '(none)'){
      LOG("Pick an export Date first (ROI Grid section).");
      return;
    }
    if(!ds){
      LOG("Pick a Dataset first (ROI Grid section).");
      return;
    }

    const url = `/sbr_table_json?date=${encodeURIComponent(date)}&dataset=${encodeURIComponent(ds)}&t=${Date.now()}`;
    const j = await (await fetch(url)).json();
    if(!j.ok){
      LOG(j.msg || "Failed to load SBR table.");
      if(sbrTablePanel) sbrTablePanel.style.display = "none";
      return;
    }

    if(sbrTableMeta) sbrTableMeta.textContent = `Date: ${j.date} · Dataset: ${j.dataset}`;
    if(sbrTableBody) sbrTableBody.innerHTML = renderSbrTable(j.table || {});
    if(sbrTablePanel) sbrTablePanel.style.display = "block";
  }catch(e){
    LOG("loadSbrTable error: " + e);
  }
}

const loadSbrTableBtn = document.getElementById('loadSbrTable');
if(loadSbrTableBtn) loadSbrTableBtn.onclick = loadSbrTable;

/* =========================
   Positivity Threshold (neg only)
   ========================= */
function mean(arr){
  const xs = (arr || []).filter(v => Number.isFinite(v));
  if(!xs.length) return NaN;
  return xs.reduce((a,b)=>a+b,0) / xs.length;
}
function stdSample(arr){
  const xs = (arr || []).filter(v => Number.isFinite(v));
  const n = xs.length;
  if(n < 2) return NaN;
  const m = mean(xs);
  const v = xs.reduce((acc,x)=>acc + (x-m)*(x-m), 0) / (n - 1);
  return Math.sqrt(v);
}

async function computePositivityThresholdNeg(){
  const date = getExportDate();
  const ds = getExportDatasetFamily();
  if(!date || date === '(none)'){
    LOG("Pick an export Date first (ROI Grid section).");
    return;
  }
  if(!ds){
    LOG("Pick a Dataset first.");
    return;
  }

  const url = `/sbr_table_json?date=${encodeURIComponent(date)}&dataset=${encodeURIComponent(ds)}&t=${Date.now()}`;
  const j = await (await fetch(url)).json();
  if(!j.ok){
    LOG(j.msg || "Failed to load SBR table for threshold.");
    return;
  }

  const table = j.table || {};
  const cells = table.cells || {};
  const negRow = cells["neg"] || cells["NEG"] || null;

  const repLines = [];
  const sbrs = [];
  for(const rep of ["N1","N2","N3"]){
    const cell = negRow ? negRow[rep] : null;
    if(cell && cell.ok){
      const s = Number(cell.sbr);
      repLines.push(`${rep}: sbr=${fmt6(s)}`);
      if(Number.isFinite(s)) sbrs.push(s);
    } else {
      repLines.push(`${rep}: (missing)`);
    }
  }

  const mu = mean(sbrs);
  const sigma = stdSample(sbrs);
  const thr = (Number.isFinite(mu) && Number.isFinite(sigma)) ? (mu + 3*sigma) : NaN;

  const panel = document.getElementById('ptPanel');
  const body = document.getElementById('ptBody');
  if(panel) panel.style.display = 'block';

  const muEl = document.getElementById('ptMu');
  const sgEl = document.getElementById('ptSigma');
  const thEl = document.getElementById('ptThr');
  if(muEl) muEl.textContent = fmt6(mu);
  if(sgEl) sgEl.textContent = fmt6(sigma);
  if(thEl) thEl.textContent = fmt6(thr);

  const canCompute = (sbrs.length >= 2) && Number.isFinite(mu) && Number.isFinite(sigma);
  const eq1 = "PositivityThreshold = μ + 3σ";
  let eq2 = "";
  let eq3 = "";

  if(canCompute){
    eq2 = `μ = mean(${sbrs.map(fmt6).join(", ")}) = ${fmt6(mu)}\n` +
          `σ = std(${sbrs.map(fmt6).join(", ")}) = ${fmt6(sigma)}`;
    eq3 = `PositivityThreshold = ${fmt6(mu)} + 3·${fmt6(sigma)} = ${fmt6(thr)}`;
  } else {
    eq2 = `Need at least 2 replicate SBR values to compute σ.\nAvailable: ${sbrs.length} valid replicate(s).`;
    eq3 = "";
  }

  if(body){
    body.innerHTML = `
      <div class="muted">Date: ${j.date} · Dataset family: ${j.dataset} · Row: <span class="mono">neg</span></div>
      <div class="mono" style="margin-top:10px;white-space:pre-wrap">
Negative-control replicate SBRs
${repLines.join("\n")}

${eq1}
${eq2}
${eq3}
      </div>
    `;
  }
}

/* ========================= */

IMG.style.userSelect = 'none';
IMG.style.touchAction = 'none';

function hardResetROI(){
  RECT.style.display='none';
  RECT.style.left = RECT.style.top = '0px';
  RECT.style.width = RECT.style.height = '0px';
  mouseDown = false; roiMode = false;
  IMG.style.cursor = 'default';
}

function setImgSrc() {
  if (!currentDataset || currentIndex===null) return;
  if (imgBusy) return;
  imgBusy = true;
  IMG.classList.add('loading');
  const url = `/image?dataset=${encodeURIComponent(currentDataset)}&index=${currentIndex}&t=${Date.now()}`;
  const done = ()=>{ imgBusy=false; IMG.classList.remove('loading'); };
  IMG.onload = done; IMG.onerror = done;
  IMG.src = url;
}

async function fetchDatasets(){
  const r = await fetch('/datasets'); const j = await r.json();
  const opts = j.names.map(n=>`<option>${n}</option>`).join('');
  DS.innerHTML = opts;
  GRID_DS.innerHTML = opts;
  LOG("Datasets: " + j.names.join(", "));
}

async function loadDataset(){
  currentDataset = DS.value; currentIndex = null;
  const r = await fetch('/images?dataset=' + encodeURIComponent(currentDataset));
  const j = await r.json();
  if (!j.files.length){ IMGSEL.innerHTML=""; IMG.removeAttribute('src'); ROWNAME.value=""; LOG("No images."); return; }

  IMGSEL.innerHTML = j.files.map((f,i)=>`<option value="${i}">${f.filename} (${f.width}×${f.height})</option>`).join('');
  hardResetROI();
  currentIndex = 0;
  ROWNAME.value = j.files[0].row;
  setImgSrc();
  LOG(`Loaded dataset: ${currentDataset} (${j.files.length} files)`);

  const date = getExportDate();
  await loadDfFor(currentDataset, date);
  await fetchDfRows();
}

IMGSEL.onchange = async e => {
  currentIndex = parseInt(e.target.value,10);
  hardResetROI();
  const r = await fetch(`/row_name?dataset=${encodeURIComponent(currentDataset)}&index=${currentIndex}`);
  const j = await r.json();
  ROWNAME.value = j.row || '';
  setImgSrc();
};

document.getElementById('load').onclick = loadDataset;

document.getElementById('select').onclick = () => {
  roiMode = !roiMode;
  RECT.style.display = roiMode ? 'block' : 'none';
  IMG.style.cursor = roiMode ? 'crosshair' : 'default';
  LOG("ROI: " + (roiMode ? "ON" : "OFF"));
};

document.getElementById('reset').onclick  = async () => {
  hardResetROI();
  try{
    const r = await fetch('/reset_view', {method:'POST'}); const j = await r.json(); LOG(j.msg || 'Reset.');
  }catch(e){ LOG('Reset failed: ' + e); }
  setImgSrc();
};

document.getElementById('rotate').onclick = async () => {
  const r = await fetch('/rotate', {method:'POST'}); const j = await r.json(); LOG(j.msg); setImgSrc();
};

document.getElementById('applyGray').onclick = async () => {
  try{
    const r = await fetch('/apply_grayscale', {method:'POST'});
    const j = await r.json();
    LOG(j.msg || "Applied grayscale.");
  }catch(e){
    LOG("Apply grayscale failed: " + e);
  }
  setImgSrc();
};

function clamp(v,min,max){ return Math.max(min, Math.min(max, v)); }

// Returns:
// - x,y relative to the image content area (0..imgRect.width/height)
// - imgRect and wrapRect
// - dx,dy = image's top-left offset inside the wrapper (in CSS pixels)
function posInImg(e){
  const imgRect = IMG.getBoundingClientRect();
  const wrapRect = IMG.parentElement.getBoundingClientRect(); // .imgWrap

  const x = clamp(e.clientX - imgRect.left, 0, imgRect.width);
  const y = clamp(e.clientY - imgRect.top , 0, imgRect.height);

  const dx = imgRect.left - wrapRect.left;
  const dy = imgRect.top  - wrapRect.top;

  return { x, y, imgRect, wrapRect, dx, dy };
}


IMG.addEventListener('dragstart', e => e.preventDefault());
IMG.addEventListener('mousedown', e=>{
  if(!roiMode) return;
  e.preventDefault();

  const p = posInImg(e);
  mouseDown = true;
  startX = p.x;
  startY = p.y;

  // IMPORTANT: add dx/dy so the rect is positioned in wrapper-space
  RECT.style.left = (p.dx + startX) + 'px';
  RECT.style.top  = (p.dy + startY) + 'px';
  RECT.style.width = '0px';
  RECT.style.height = '0px';
});


window.addEventListener('mousemove', e=>{
  if(!roiMode || !mouseDown) return;

  const p = posInImg(e);
  const left = Math.min(startX, p.x);
  const top  = Math.min(startY, p.y);
  const w = Math.abs(p.x - startX);
  const h = Math.abs(p.y - startY);

  RECT.style.left = (p.dx + left) + 'px';
  RECT.style.top  = (p.dy + top) + 'px';
  RECT.style.width = w + 'px';
  RECT.style.height = h + 'px';
});

window.addEventListener('mouseup', async e=>{
  if(!roiMode || !mouseDown) return;
  mouseDown = false;

  const p = posInImg(e);
  const r = p.imgRect; // image rect

  const left = Math.min(startX, p.x);
  const top  = Math.min(startY, p.y);
  const w = Math.abs(p.x - startX);
  const h = Math.abs(p.y - startY);

  if (w < 3 || h < 3) { LOG("Selection too small."); return; }

  // normalized selection relative to IMAGE (correct)
  const nx = left / r.width, ny = top / r.height, nw = w / r.width, nh = h / r.height;

  const tw = Math.round(r.width), th = Math.round(r.height);

  const resp = await fetch('/apply_crop_zoom', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body: JSON.stringify({nx, ny, nw, nh, tw, th})
  });
  const j = await resp.json();
  LOG(j.msg);
  setImgSrc();
});


document.getElementById('save').onclick = async () => {
  if (!currentDataset || currentIndex===null) return;
  const r = await fetch('/save_roi', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body: JSON.stringify({ dataset: currentDataset, index: currentIndex })
  });
  const j = await r.json(); LOG(j.msg);
};

document.getElementById('export').onclick = async () => {
  if (!currentDataset) return;
  const r = await fetch('/export_df', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body: JSON.stringify({ dataset: currentDataset })
  });
  const j = await r.json();
  LOG(j.msg);

  const date = getExportDate();
  const gridDs = getExportDatasetFamily();
  await loadDfFor(gridDs || currentDataset, date);
  await fetchDfRows();
};

// dates + grid
const DFGRID  = document.getElementById('dfGrid');
const DFDATES = document.getElementById('dfDate');

async function fetchDfDates(){
  const r = await fetch('/exports'); const j = await r.json();
  if (!j.dates.length){ DFDATES.innerHTML = '<option>(none)</option>'; DFGRID.style.display='none'; return; }
  DFDATES.innerHTML = j.dates.map(d=>`<option>${d}</option>`).join('');
}
document.getElementById('refreshDfDates').onclick = fetchDfDates;

document.getElementById('plotGrid').onclick = async () => {
  const date = DFDATES.value;
  const ds = GRID_DS.value;
  const dip = (document.getElementById('gridDip')?.value || '0');
  if(!date || date === '(none)'){ LOG('No export dates found.'); return; }
  if(!ds){ LOG('Pick a dataset.'); return; }

  await loadDfFor(ds, date);
  await fetchDfRows();

  DFGRID.src = `/analyze_grid?date=${encodeURIComponent(date)}&dataset=${encodeURIComponent(ds)}&include_dip=${encodeURIComponent(dip)}&t=${Date.now()}`;
  DFGRID.style.display = 'block';

  if(sbrTablePanel) sbrTablePanel.style.display = 'block';
};

// ----------------- interactive SBR -----------------
const canvas = document.getElementById('sbrCanvas');
const ctx = canvas.getContext('2d');
const blueInfo = document.getElementById('blueInfo');
const redInfo = document.getElementById('redInfo');
const greenInfo = document.getElementById('greenInfo');
const yellowInfo = document.getElementById('yellowInfo');

const sbrRowSel = document.getElementById('sbrRow');
const sbrRepSel = document.getElementById('sbrRep');
let sbrProfile = null;
let sbrN = 0;
let xBlue = null;
let xRed = null;
let xPeak = null;
let peakVal = null;
let baselineVal = null;
let baselineXHalf = null;
let sbrRowName = null;

let plotGeom = null;

// NEW: dynamic baseline overlay (line segment) + baseline@peak
let baselineLine = null;    // {x1,y1,x2,y2}
let baselineAtPeak = null;  // number

function setCanvasSize(){
  const rect = canvas.getBoundingClientRect();
  const dpr = window.devicePixelRatio || 1;
  canvas.width = Math.max(1, Math.floor(rect.width * dpr));
  canvas.height = Math.max(1, Math.floor(rect.height * dpr));
  ctx.setTransform(dpr,0,0,dpr,0,0);
}

function resetLines(){
  xBlue = null; xRed = null; xPeak = null;
  peakVal = null;
  baselineVal = null;
  baselineXHalf = null;

  baselineLine = null;
  baselineAtPeak = null;

  blueInfo.textContent = "-";
  redInfo.textContent = "-";
  greenInfo.textContent = "-";
  yellowInfo.textContent = "-";
  MSG("Click two points on the graph (x must be >=5 and <= n-6).");
  drawSBR();
}
document.getElementById('resetLines').onclick = resetLines;

async function fetchDfRows(){
  try{
    const r = await fetch('/df_rows'); const j = await r.json();
    if(!j.ok){ sbrRowSel.innerHTML = '<option>(export first)</option>'; return; }
    const rows = j.rows || [];
    if(!rows.length){ sbrRowSel.innerHTML = '<option>(no rows)</option>'; return; }
    sbrRowSel.innerHTML = rows.map(x=>`<option value="${x}">${x}</option>`).join('');
  }catch(e){
    sbrRowSel.innerHTML = '<option>(error)</option>';
  }
}

function niceNum(range, round) {
  const exponent = Math.floor(Math.log10(range || 1));
  const fraction = range / Math.pow(10, exponent);
  let niceFraction;
  if (round) {
    if (fraction < 1.5) niceFraction = 1;
    else if (fraction < 3) niceFraction = 2;
    else if (fraction < 7) niceFraction = 5;
    else niceFraction = 10;
  } else {
    if (fraction <= 1) niceFraction = 1;
    else if (fraction <= 2) niceFraction = 2;
    else if (fraction <= 5) niceFraction = 5;
    else niceFraction = 10;
  }
  return niceFraction * Math.pow(10, exponent);
}

function computeTicks(min, max, maxTicks = 5) {
  const range = niceNum(max - min, false);
  const step = niceNum(range / Math.max(1, (maxTicks - 1)), true);
  const graphMin = Math.floor(min / step) * step;
  const graphMax = Math.ceil(max / step) * step;

  const ticks = [];
  for (let v = graphMin; v <= graphMax + 0.5 * step; v += step) ticks.push(v);
  return { ticks, graphMin, graphMax, step };
}

function formatTick(v, step){
  const absStep = Math.abs(step || 0);
  let decimals = 0;
  if (absStep > 0) {
    decimals = Math.max(0, -Math.floor(Math.log10(absStep)) + 1);
    decimals = Math.min(decimals, 6);
  }
  let s = v.toFixed(decimals);
  s = s.replace(/\.?0+$/, '');
  return s;
}

function drawSBR(){
  if (!sbrProfile || !sbrProfile.length){
    canvas.style.display = 'none';
    plotGeom = null;
    return;
  }
  canvas.style.display = 'block';
  setCanvasSize();

  const rect = canvas.getBoundingClientRect();
  const W = rect.width;
  const H = rect.height;
  ctx.clearRect(0,0,W,H);

  const padL = 62, padR = 18, padT = 34, padB = 46;

  let yMin = Infinity, yMax = -Infinity;
  for (const v of sbrProfile){ if (v<yMin) yMin=v; if (v>yMax) yMax=v; }
  if (!isFinite(yMin) || !isFinite(yMax)){ yMin=0; yMax=1; }

  // include baseline line endpoints + baseline@peak in y-range so it's visible
  if (baselineLine){
    yMin = Math.min(yMin, Number(baselineLine.y1), Number(baselineLine.y2));
    yMax = Math.max(yMax, Number(baselineLine.y1), Number(baselineLine.y2));
  }
  if (baselineAtPeak !== null && isFinite(baselineAtPeak)){
    yMin = Math.min(yMin, baselineAtPeak);
    yMax = Math.max(yMax, baselineAtPeak);
  }

  const yPad = (yMax - yMin) * 0.08 || 0.01;
  yMin -= yPad; yMax += yPad;

  const maxX = sbrN - 1;
  const xTicks = [];
  const xStep = 50;
  for (let x=0; x<=maxX; x+=xStep) xTicks.push(x);
  if (xTicks[xTicks.length-1] !== maxX) xTicks.push(maxX);

  const yt = computeTicks(yMin, yMax, 5);

  const xToPx = (x) => padL + (x / (sbrN - 1)) * (W - padL - padR);
  const yToPx = (y) => (H - padB) - ((y - yt.graphMin) / (yt.graphMax - yt.graphMin + 1e-9)) * (H - padT - padB);

  plotGeom = { padL, padR, padT, padB, W, H, xToPx, yToPx };

  ctx.lineWidth = 1;
  ctx.strokeStyle = "rgba(230,237,243,0.18)";
  ctx.font = "12px ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace";
  ctx.fillStyle = "#e6edf3";

  ctx.textAlign = "center";
  ctx.textBaseline = "top";
  for (const xtv of xTicks){
    const x = xToPx(xtv);
    ctx.beginPath();
    ctx.moveTo(x, padT);
    ctx.lineTo(x, H-padB);
    ctx.stroke();
    ctx.fillText(String(xtv), x, H-padB+8);
  }

  ctx.textAlign = "right";
  ctx.textBaseline = "middle";
  for (const ytv of yt.ticks){
    const y = yToPx(ytv);
    ctx.beginPath();
    ctx.moveTo(padL, y);
    ctx.lineTo(W-padR, y);
    ctx.stroke();
    ctx.fillText(formatTick(ytv, yt.step), padL-10, y);
  }

  ctx.strokeStyle = "rgba(230,237,243,0.35)";
  ctx.lineWidth = 1.25;
  ctx.beginPath();
  ctx.rect(padL, padT, (W-padL-padR), (H-padT-padB));
  ctx.stroke();

  ctx.fillStyle = "#e6edf3";
  ctx.font = "600 16px system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,sans-serif";
  const repLabel = (sbrRepSel && sbrRepSel.value) ? ` · ${sbrRepSel.value}` : "";
  const title = `SBR – ${(getExportDatasetFamily() || window.DF_DATASET || "")}${repLabel} – row ${sbrRowName || ""}`;
  ctx.textAlign = "center";
  ctx.textBaseline = "top";
  ctx.fillText(title, W/2, 8);

  ctx.fillStyle = "#e6edf3";
  ctx.font = "13px system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,sans-serif";
  ctx.textAlign = "center";
  ctx.textBaseline = "top";
  ctx.fillText("Position along strip", (padL + (W-padR))/2, H-28);

  ctx.save();
  ctx.translate(18, (padT + (H-padB))/2);
  ctx.rotate(-Math.PI/2);
  ctx.textAlign = "center";
  ctx.textBaseline = "top";
  ctx.fillText("Intensity (smoothed)", 0, 0);
  ctx.restore();

  function shadeWindow(ix, rgba){
    if (ix === null) return;
    const left = xToPx(ix - 5);
    const right = xToPx(ix + 5);
    ctx.fillStyle = rgba;
    ctx.fillRect(left, padT, (right-left), (H-padB-padT));
  }
  shadeWindow(xBlue, "rgba(59,130,246,0.12)");
  shadeWindow(xRed,  "rgba(239,68,68,0.12)");

  // DRAW BASELINE LINE SEGMENT (dynamic) + baseline@peak marker
  if (baselineLine){
    const x1 = xToPx(Number(baselineLine.x1));
    const y1 = yToPx(Number(baselineLine.y1));
    const x2 = xToPx(Number(baselineLine.x2));
    const y2 = yToPx(Number(baselineLine.y2));

    ctx.strokeStyle = "#eab308";
    ctx.lineWidth = 2;
    ctx.beginPath();
    ctx.moveTo(x1, y1);
    ctx.lineTo(x2, y2);
    ctx.stroke();

    if (xPeak !== null && Number.isFinite(baselineAtPeak)){
      const xp = xToPx(Number(xPeak));
      const yp = yToPx(Number(baselineAtPeak));
      ctx.fillStyle = "#eab308";
      ctx.beginPath();
      ctx.arc(xp, yp, 4, 0, Math.PI*2);
      ctx.fill();
    }
  }

  // profile curve
  ctx.strokeStyle = "#9bbcff";
  ctx.lineWidth = 2;
  ctx.beginPath();
  for (let i=0;i<sbrN;i++){
    const x = xToPx(i);
    const y = yToPx(sbrProfile[i]);
    if (i===0) ctx.moveTo(x,y); else ctx.lineTo(x,y);
  }
  ctx.stroke();

  function drawV(ix, color){
    const x = xToPx(ix);
    ctx.strokeStyle = color;
    ctx.lineWidth = 2;
    ctx.beginPath();
    ctx.moveTo(x, padT);
    ctx.lineTo(x, H-padB);
    ctx.stroke();
  }
  if (xBlue !== null) drawV(xBlue, "#3b82f6");
  if (xRed !== null)  drawV(xRed,  "#ef4444");
  if (xPeak !== null) drawV(xPeak, "#22c55e");

  ctx.font = "12px system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,sans-serif";
  ctx.textAlign = "left";
  ctx.textBaseline = "top";
  let lx = padL + 8, ly = padT + 8;
  if (xBlue !== null){
    ctx.fillStyle = "#3b82f6"; ctx.fillRect(lx, ly+4, 10, 2);
    ctx.fillStyle = "#e6edf3"; ctx.fillText(`Blue @ ${xBlue} (±5)`, lx+14, ly);
    ly += 18;
  }
  if (xRed !== null){
    ctx.fillStyle = "#ef4444"; ctx.fillRect(lx, ly+4, 10, 2);
    ctx.fillStyle = "#e6edf3"; ctx.fillText(`Red @ ${xRed} (±5)`, lx+14, ly);
    ly += 18;
  }
  if (xPeak !== null){
    ctx.fillStyle = "#22c55e"; ctx.fillRect(lx, ly+4, 10, 2);
    ctx.fillStyle = "#e6edf3"; ctx.fillText(`PeakX @ ${xPeak} (sig ${fmt6(peakVal)})`, lx+14, ly);
    ly += 18;
  }
  if (baselineVal !== null && isFinite(baselineVal)){
    ctx.fillStyle = "#eab308"; ctx.fillRect(lx, ly+4, 10, 2);
    ctx.fillStyle = "#e6edf3"; ctx.fillText(`Baseline@Peak = ${fmt6(baselineVal)} (@x=${baselineXHalf})`, lx+14, ly);
    ly += 18;
    if (baselineLine){
      ctx.fillStyle = "#eab308"; ctx.fillRect(lx, ly+4, 10, 2);
      ctx.fillStyle = "#e6edf3"; ctx.fillText(`Baseline line (between snapped endpoints)`, lx+14, ly);
    }
  }
}

function pxToIndex(clientX){
  if (!plotGeom) return 0;
  const rect = canvas.getBoundingClientRect();
  const { padL, padR, W } = plotGeom;
  const x = Math.max(padL, Math.min(W-padR, clientX - rect.left));
  const t = (x - padL) / (W - padL - padR);
  return Math.round(t * (sbrN - 1));
}

function validateIndex(ix){
  if (ix < 5 || ix > (sbrN - 6)){
    return {ok:false, msg:`Pick a different point: need 5 points on each side (valid: 5..${sbrN-6}). You picked ${ix}.`};
  }
  return {ok:true, msg:""};
}

canvas.addEventListener('click', async (e)=>{
  if (!sbrProfile) return;
  const ix = pxToIndex(e.clientX);
  const v = validateIndex(ix);
  if (!v.ok){ MSG(v.msg); return; }

  if (xBlue === null){
    xBlue = ix;
    blueInfo.textContent = `${ix} (window ${ix-5}-${ix+5})`;
    MSG("Blue set. Click second point for Red.");
    drawSBR();
    return;
  }
  if (xRed === null){
    xRed = ix;
    redInfo.textContent = `${ix} (window ${ix-5}-${ix+5})`;
    drawSBR();
    MSG("Red set. Computing baseline line + saving into CSV...");

    const row = sbrRowSel.value || "";
    const idx = document.getElementById('sbrIndex').value || '0';
    const rep = getSbrReplicate();
    const date = getExportDate();
    const dsFamily = getExportDatasetFamily();

    const r = await fetch('/update_medians', {
      method:'POST',
      headers:{'Content-Type':'application/json'},
      body: JSON.stringify({
        row: row,
        index: parseInt(idx,10),
        x_blue: xBlue,
        x_red: xRed,
        replicate: rep,
        date: date,
        dataset: dsFamily
      })
    });
    const j = await r.json();

    if (j.ok){
      // NEW: use overlay from backend for drawing the baseline segment correctly
      const ov = j.overlay || {};
      xPeak = Number(ov.peak_x);
      peakVal = Number(ov.peak_y);

      baselineVal = Number(ov.baseline_at_peak);
      baselineAtPeak = baselineVal;
      baselineLine = ov.baseline_line || null;

      baselineXHalf = Number.isFinite(xPeak) ? xPeak : null;

      greenInfo.textContent = Number.isFinite(xPeak) ? `${xPeak} (sig ${fmt6(peakVal)})` : "-";
      yellowInfo.textContent = Number.isFinite(baselineVal) ? `${fmt6(baselineVal)} (@x=${baselineXHalf})` : "-";
      drawSBR();

      await loadSbrTable();
      if ((sbrRowName || "").toLowerCase() === "neg") {
        await computePositivityThresholdNeg();
      }
    }

    MSG(j.msg || (j.ok ? "Updated CSV." : "Failed."));
    return;
  }

  MSG("You already selected 2 points. Click Reset Lines to choose different ones.");
});

async function loadSbrFromSelection(){
  const date = getExportDate();
  const dsFamily = getExportDatasetFamily();
  if(!date || date === '(none)'){
    MSG("Pick an export Date in the ROI Grid section first.");
    return;
  }
  if(!dsFamily){
    MSG("Pick a Dataset in the ROI Grid section first.");
    return;
  }

  await loadDfFor(dsFamily, date);

  const row = sbrRowSel.value || "";
  const idx = document.getElementById('sbrIndex').value || '0';
  const rep = getSbrReplicate();

  const url =
    `/sbr_profile_json?date=${encodeURIComponent(date)}&dataset=${encodeURIComponent(dsFamily)}` +
    `&row=${encodeURIComponent(row)}&index=${encodeURIComponent(idx)}&replicate=${encodeURIComponent(rep)}&t=${Date.now()}`;

  const r = await fetch(url);
  const j = await r.json();
  if (!j.ok){
    MSG(j.msg || "Failed to load SBR. Make sure ROIs were saved for this Date/Dataset and you exported CSV.");
    canvas.style.display = 'none';
    plotGeom = null;
    return;
  }

  window.DF_DATASET = j.dataset || dsFamily;

  sbrProfile = j.profile;
  sbrN = j.n;
  sbrRowName = j.row || row;
  resetLines();

  if(sbrTablePanel) sbrTablePanel.style.display = 'block';

  if ((sbrRowName || "").toLowerCase() === "neg") {
    const p = document.getElementById('ptPanel');
    if(p) p.style.display = 'block';
  }
}

document.getElementById('plotSBR').onclick = loadSbrFromSelection;

if (sbrRepSel){
  sbrRepSel.onchange = async () => {
    if (sbrProfile){
      await loadSbrFromSelection();
    }
  };
}

/* ============================================================
   Demo viewer (UPDATED: works without manual xBlue/xRed clicks)
   ============================================================ */
const demoMsg = document.getElementById('demoMsg');
const demoStrip = document.getElementById('demoStrip');
const demoAverage = document.getElementById('demoAverage');
const demoJson = document.getElementById('demoJson');
const demoRowStats = document.getElementById('demoRowStats');
const demoPooledStats = document.getElementById('demoPooledStats');
const expPlot = document.getElementById('expPlot');

function setDemoMsg(m){ demoMsg.textContent = m || ""; }

function buildDemoUrl(){
  const row = sbrRowSel.value || "";
  const idx = document.getElementById('sbrIndex').value || '0';
  const rep = getSbrReplicate();
  const date = getExportDate();
  const dsFamily = getExportDatasetFamily();

  let url =
    `/demo_full_json?date=${encodeURIComponent(date)}&dataset=${encodeURIComponent(dsFamily)}` +
    `&row=${encodeURIComponent(row)}&index=${encodeURIComponent(idx)}&replicate=${encodeURIComponent(rep)}`;

  // If user clicked blue/red this session, pass them. Otherwise backend falls back to CSV saved x_blue/x_red.
  if (xBlue !== null && xRed !== null){
    url += `&x_blue=${xBlue}&x_red=${xRed}`;
  }
  url += `&t=${Date.now()}`;
  return url;
}

async function runDemoPlots(){
  const rep = getSbrReplicate();
  const date = getExportDate();
  const dsFamily = getExportDatasetFamily();

  if(!date || date === '(none)'){
    setDemoMsg("Pick an export Date (ROI Grid section) first.");
    return;
  }
  if(!dsFamily){
    setDemoMsg("Pick a Dataset (ROI Grid section) first.");
    return;
  }

  const j = await (await fetch(buildDemoUrl())).json();
  if (!j.ok){
    setDemoMsg(j.msg || "Demo failed.");
    return;
  }

  demoRowStats.textContent =
    `${Number(j.demo.row_ratio_mean).toFixed(4)} ± ${Number(j.demo.row_ratio_sem).toFixed(4)} (n=${j.demo.row_ratio_n})`;

  const rowName = j.row_name || j.row || (sbrRowSel.value || "");

  demoStrip.src =
    `/demo_strip.png?date=${encodeURIComponent(date)}&dataset=${encodeURIComponent(dsFamily)}` +
    `&row=${encodeURIComponent(rowName)}&replicate=${encodeURIComponent(rep)}&t=${Date.now()}`;

  demoAverage.src =
    `/demo_average.png?date=${encodeURIComponent(date)}&dataset=${encodeURIComponent(dsFamily)}` +
    `&row=${encodeURIComponent(rowName)}&replicate=${encodeURIComponent(rep)}&t=${Date.now()}`;

  demoStrip.style.display = 'block';
  demoAverage.style.display = 'block';

  setDemoMsg("Demo plots updated.");
}

document.getElementById('runDemo').onclick = runDemoPlots;

document.getElementById('loadDemoJson').onclick = async ()=>{
  const date = getExportDate();
  const dsFamily = getExportDatasetFamily();
  if(!date || date === '(none)'){
    setDemoMsg("Pick an export Date (ROI Grid section) first.");
    return;
  }
  if(!dsFamily){
    setDemoMsg("Pick a Dataset (ROI Grid section) first.");
    return;
  }

  const r = await fetch(buildDemoUrl());
  const j = await r.json();
  if (!j.ok){
    setDemoMsg(j.msg || "Demo JSON failed.");
    return;
  }
  demoJson.textContent = JSON.stringify(j, null, 2);
  setDemoMsg("Raw demo JSON loaded (includes results.R_all).");
};

document.getElementById('addExp').onclick = async ()=>{
  const date = getExportDate();
  const dsFamily = getExportDatasetFamily();
  if(!date || date === '(none)'){
    setDemoMsg("Pick an export Date (ROI Grid section) first.");
    return;
  }
  if(!dsFamily){
    setDemoMsg("Pick a Dataset (ROI Grid section) first.");
    return;
  }

  const row = sbrRowSel.value || "";
  const rep = getSbrReplicate();

  const payload = { row: row, replicate: rep, date: date, dataset: dsFamily };
  if (xBlue !== null && xRed !== null){
    payload.x_blue = xBlue;
    payload.x_red = xRed;
  }

  const r = await fetch('/experiment_add', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body: JSON.stringify(payload)
  });
  const out = await r.json();
  if (!out.ok){
    setDemoMsg(out.msg || "Failed to add experiment.");
    return;
  }
  demoPooledStats.textContent =
    `${Number(out.R_best).toFixed(4)} ± ${Number(out.SE_best).toFixed(4)} (N=${out.N_total})`;
  expPlot.src = `/experiment_plot.png?t=${Date.now()}`;
  expPlot.style.display = 'block';
  setDemoMsg(out.msg || "Experiment added.");
};

document.getElementById('clearExp').onclick = async ()=>{
  const r = await fetch('/experiment_clear', {method:'POST'});
  const j = await r.json();
  demoPooledStats.textContent = "-";
  expPlot.style.display = 'none';
  setDemoMsg(j.msg || "Cleared.");
};

// ----------------- Final graph (unchanged) -----------------
const finalGraph = document.getElementById('finalGraph');
const finalMsg = document.getElementById('finalMsg');

document.getElementById('plotFinal').onclick = () => {
  const mode = (document.getElementById('finalMode')?.value || 'combined').toUpperCase();
  const srcLegacy = document.getElementById('finalSource').value || "df";

  const date = getExportDate();
  const ds = getExportDatasetFamily();

  if (!date || date === '(none)') {
    finalMsg.textContent = "Pick an export Date (ROI Grid section) first.";
    return;
  }
  if (!ds) {
    finalMsg.textContent = "Pick a Dataset (ROI Grid section) first.";
    return;
  }

  finalMsg.textContent = "Rendering...";

  let url = "";
  if (mode === 'COMBINED') {
    url = `/final_graph_combined.png?date=${encodeURIComponent(date)}&dataset=${encodeURIComponent(ds)}&t=${Date.now()}`;
  } else if (mode === 'N1' || mode === 'N2' || mode === 'N3') {
    url = `/final_graph_selected.png?date=${encodeURIComponent(date)}&dataset=${encodeURIComponent(ds)}&mode=${encodeURIComponent(mode)}&t=${Date.now()}`;
  } else {
    url = `/final_graph.png?source=${encodeURIComponent(srcLegacy)}&t=${Date.now()}`;
  }

  finalGraph.src = url;
  finalGraph.style.display = 'block';
  finalGraph.onload = () => { finalMsg.textContent = ""; };
  finalGraph.onerror = () => {
    finalMsg.textContent =
      "Failed to load final graph. Ensure the endpoint exists and CSV exports exist for the selected date/dataset.";
  };
};

// ----------------- init -----------------
function init() {
  fillSavePath();
  fetchDatasets()
    .then(fetchDfDates)
    .then(async ()=>{
      const date = getExportDate();
      const ds = getExportDatasetFamily();
      if(date && date !== '(none)' && ds){
        await loadDfFor(ds, date);
      }
    })
    .then(fetchDfRows)
    .catch(e => LOG("Init error: " + e));

  const btn = document.getElementById('computePT');
  if(btn) btn.onclick = computePositivityThresholdNeg;

  if(sbrTablePanel) sbrTablePanel.style.display = 'block';
}
window.addEventListener('load', init);
window.addEventListener('resize', () => { if (sbrProfile) drawSBR(); });

</script>
</body></html>
"""

# ---- final assembled template ----
TEMPLATE_HTML = TEMPLATE_HTML_1 + TEMPLATE_HTML_2


In [ ]:
# /content/uwlfa_gui_cells/cell6_state_and_helpers.py

from __future__ import annotations

import io
import re
from pathlib import Path
from typing import Dict, Tuple, Optional, List, Callable, Any

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

# ----------------- server state -----------------
CURRENT_IMAGE = None
ORIGINAL_IMAGE = None

TARGET_H, TARGET_W = 70, 270

DISPLAY_INVERT_INTENSITY = False  # affects /image preview only
ANALYSIS_INVERT_INTENSITY = True  # affects ROI saving + analysis

ROI_SAVE_COUNT: Dict[Tuple[str, str], int] = {}
ROI_LATEST_PATH: Dict[Tuple[str, str], Path] = {}
ROI_LATEST_SHAPE: Dict[Tuple[str, str], Tuple[int, int]] = {}

DF_GRAY: Optional[pd.DataFrame] = None
DF_DATASET: Optional[str] = None
DF_CSV_PATH: Optional[Path] = None

EXP_R_LIST: List[np.ndarray] = []
EXP_LABELS: List[str] = []

# =========================
# Graphing.ipynb-style helpers
# =========================
EXCLUDE_ROWS_GRAPH_FINAL = {"cc", "k", "dip", "dips"}
GRAPH_ROW_ORDER = ["neg", "1e5", "1.5e5", "5e5", "1e6", "5e6", "1e7", "pos"]


def _normalize_row_label(row: str) -> str:
    r = (row or "").strip().lower()
    if r == "dips":
        r = "dip"
    if r.startswith("neg"):
        r = "neg"
    if r.startswith("pos"):
        r = "pos"
    return r


def _replicate_from_dataset_name(ds_raw: str) -> Optional[str]:
    s = (ds_raw or "").strip().lower()

    m = re.search(r"_n *= *([123])", s)
    if m:
        return f"N{m.group(1)}"

    m = re.search(r"(^|[^0-9a-z])n([123])([^0-9a-z]|$)", s)
    if m:
        return f"N{m.group(2)}"

    return None


def _row_to_bacterial_load(row: str) -> Optional[float]:
    r = _normalize_row_label(row)
    if not r:
        return None
    if r in {"neg", "negative"}:
        return 0.0
    if r in {"pos", "positive"}:
        return 1e9
    if r in EXCLUDE_ROWS_GRAPH_FINAL:
        return None

    m = re.match(r"^([0-9]+(?:[.][0-9]+)?)e([0-9]+)$", r)
    if m:
        return float(float(m.group(1)) * (10 ** float(m.group(2))))
    return None


# =========================
# Image pipeline + normalization
# =========================
def _dtype_max_value(arr: np.ndarray) -> float:
    if arr is None:
        return 1.0
    if arr.dtype == np.uint8:
        return 255.0
    if arr.dtype == np.uint16:
        return 65535.0
    try:
        mx = float(np.nanmax(arr))
        return mx if mx > 0 else 1.0
    except Exception:
        return 1.0


def _to_gray_like_notebook(img: np.ndarray) -> np.ndarray:
    if img is None:
        raise ValueError("No image.")
    if img.ndim == 2:
        return img.astype(np.float32)
    return np.mean(img.astype(np.float32), axis=2)


def _normalize_gray_to_01(gray: np.ndarray) -> np.ndarray:
    g = np.asarray(gray, dtype=np.float32)
    if g.size == 0 or (not np.isfinite(g).any()):
        return np.zeros_like(g, dtype=np.float32)
    if float(np.nanmax(g)) > 1.5:
        denom = _dtype_max_value(gray)
        denom = denom if denom > 0 else (float(np.nanmax(g)) if float(np.nanmax(g)) > 0 else 1.0)
        g = g / float(denom)
    return np.clip(g, 0.0, 1.0).astype(np.float32)


def _autocontrast_u8(gray01: np.ndarray, p_lo: float = 1.0, p_hi: float = 99.0) -> np.ndarray:
    g = np.asarray(gray01, dtype=np.float32)
    if g.size == 0:
        return np.zeros((1, 1), dtype=np.uint8)

    finite = g[np.isfinite(g)]
    if finite.size == 0:
        return np.zeros(g.shape, dtype=np.uint8)

    lo = float(np.percentile(finite, p_lo))
    hi = float(np.percentile(finite, p_hi))
    if (not np.isfinite(lo)) or (not np.isfinite(hi)) or hi <= lo:
        lo = float(np.min(finite))
        hi = float(np.max(finite))
        if hi <= lo:
            return (np.clip(g, 0.0, 1.0) * 255.0).astype(np.uint8)

    out = (g - lo) / (hi - lo)
    out = np.clip(out, 0.0, 1.0)
    return (out * 255.0).astype(np.uint8)


def _apply_grayscale_preserve_dtype(img: np.ndarray) -> np.ndarray:
    if img is None:
        raise ValueError("No image.")
    if img.ndim == 2:
        return img
    g = np.mean(img.astype(np.float32), axis=2)

    if img.dtype == np.uint8:
        return np.clip(g, 0, 255).astype(np.uint8)
    if img.dtype == np.uint16:
        return np.clip(g, 0, 65535).astype(np.uint16)

    return g.astype(np.float32)


def _load_image_by(dataset: str, index: int):
    files = DATASETS.get(dataset, [])
    if index < 0 or index >= len(files):
        return None
    path = files[index]["path"]
    return cv2.imread(path, cv2.IMREAD_UNCHANGED)


def _normalized_roi_from_current() -> np.ndarray:
    if CURRENT_IMAGE is None:
        raise ValueError("No current image.")

    gray = _to_gray_like_notebook(CURRENT_IMAGE)
    gray = cv2.resize(gray, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA).astype(np.float32)
    roi01 = _normalize_gray_to_01(gray)

    if ANALYSIS_INVERT_INTENSITY:
        roi01 = 1.0 - roi01

    return np.clip(roi01, 0.0, 1.0).astype(np.float32)


def _dataset_rows_from_files(ds_raw: str) -> List[str]:
    # Exclude cc/k/dip everywhere the UI uses dataset row lists
    EXCLUDE = {"cc", "k", "dip", "dips"}
    rows = [sanitize_name(row_name_from_filename(f["filename"])) for f in DATASETS.get(ds_raw, [])]
    rows = [_normalize_row_label(r) for r in rows]
    rows = [r for r in rows if r and _normalize_row_label(r) not in EXCLUDE]
    return sorted(set(rows), key=row_sort_key)


def _first_match(base_dir: Path, dataset: str, row: str) -> Optional[str]:
    folder = base_dir / dataset
    if not folder.exists():
        return None
    prefix = f"ROI_{row}_"
    for f in sorted(folder.glob(prefix + "*.tif*")):
        return str(f)
    return None


# =========================
# Notebook-aligned profile + ORIGINAL baseline-line math
# =========================
def _fmt6(x: Optional[float]) -> str:
    if x is None:
        return "nan"
    try:
        xf = float(x)
        if not np.isfinite(xf):
            return "nan"
        return f"{xf:.6f}"
    except Exception:
        return "nan"


def _roi_profile(roi: np.ndarray, *, crop_like_notebook: bool = True) -> np.ndarray:
    if roi.ndim != 2:
        raise ValueError("Expected ROI to be 2D.")
    img = roi.astype(np.float32)
    if crop_like_notebook and img.shape[0] >= 65 and img.shape[1] >= 260:
        img = img[5:65, 10:260]
    return np.mean(img, axis=0).astype(np.float32)


def snap_local(avg: np.ndarray, x_center: int, target_y: float, halfwin: int = 20) -> int:
    a = max(0, int(x_center) - int(halfwin))
    b = min(int(len(avg)) - 1, int(x_center) + int(halfwin))
    seg = avg[a : b + 1]
    rel = int(np.argmin(np.abs(seg - float(target_y))))
    return int(a + rel)


def baseline_y(x_left: int, y_left: float, x_right: int, y_right: float, x_mid: int) -> float:
    if int(x_right) == int(x_left):
        return float(y_left)
    t = (float(x_mid) - float(x_left)) / (float(x_right) - float(x_left))
    return float(y_left) + t * (float(y_right) - float(y_left))


def ui_baseline_details_from_roi(
    roi: np.ndarray,
    *,
    x_blue: int,
    x_red: int,
    crop_like_notebook: bool = True,
    snap_halfwin: int = 20,
    endpoint_median_halfwin: int = 5,
    endpoint_y_median_span: int = 10,
) -> Dict[str, object]:
    prof = _roi_profile(roi, crop_like_notebook=crop_like_notebook)
    n = int(prof.size)
    if n < 20:
        raise ValueError("Profile too short.")

    xb = int(x_blue)
    xr = int(x_red)

    hw = int(endpoint_median_halfwin)
    if xb < hw or xb > n - (hw + 1):
        raise ValueError(f"x_blue {xb} too close to edge for ±{hw} window (n={n}).")
    if xr < hw or xr > n - (hw + 1):
        raise ValueError(f"x_red {xr} too close to edge for ±{hw} window (n={n}).")

    med_blue = float(np.median(prof[xb - hw : xb + hw + 1]))
    med_red = float(np.median(prof[xr - hw : xr + hw + 1]))

    base_x_1 = int(snap_local(prof, xb, med_blue, halfwin=snap_halfwin))
    base_x_2 = int(snap_local(prof, xr, med_red, halfwin=snap_halfwin))
    if base_x_2 < base_x_1:
        base_x_1, base_x_2 = base_x_2, base_x_1

    peak_x = int(base_x_1 + int(np.argmax(prof[base_x_1 : base_x_2 + 1])))
    peak_y = float(prof[peak_x])

    y_left = float(np.median(prof[base_x_1 : min(n, base_x_1 + endpoint_y_median_span)]))
    y_right = float(np.median(prof[base_x_2 : min(n, base_x_2 + endpoint_y_median_span)]))

    baseline_at_peak = float(baseline_y(base_x_1, y_left, base_x_2, y_right, peak_x))
    sbr = float(peak_y / baseline_at_peak) if baseline_at_peak != 0 else float("inf")

    return {
        "ok": True,
        "n": n,
        "x_blue": xb,
        "x_red": xr,
        "median_blue": med_blue,
        "median_red": med_red,
        "base_x_1": base_x_1,
        "base_x_2": base_x_2,
        "y_left": y_left,
        "y_right": y_right,
        "peak_x": peak_x,
        "peak_y": peak_y,
        "baseline_at_peak": baseline_at_peak,
        "sbr": sbr,
        "baseline_line": {"x1": base_x_1, "y1": y_left, "x2": base_x_2, "y2": y_right},
        "equations": {
            "sbr_symbolic": "SBR = peak_y / baseline_at_peak",
            "sbr_substitute": f"SBR = {_fmt6(peak_y)} / {_fmt6(baseline_at_peak)}",
            "sbr_result": f"SBR = {_fmt6(sbr)}",
        },
    }


# =========================
# OPTIONAL: legacy C_Analysis method (kept)
# =========================
def _c_analysis_style_sbr_with_details(
    roi: np.ndarray,
    *,
    x_blue: int,
    x_red: int,
    crop_like_notebook: bool = True,
) -> Dict[str, object]:
    try:
        from scipy.signal import savgol_filter
    except Exception as e:
        raise RuntimeError("scipy is required for savgol_filter.") from e

    prof = _roi_profile(roi, crop_like_notebook=crop_like_notebook)
    if prof.size < 30:
        raise ValueError("Profile too short.")

    smooth = savgol_filter(prof, 5, 2)
    n = int(smooth.size)

    xb = int(x_blue)
    xr = int(x_red)
    lo = max(0, min(xb, xr))
    hi = min(n - 1, max(xb, xr))
    if hi <= lo:
        raise ValueError("Invalid x_blue/x_red range.")

    def clamp_win(s: float, e: float) -> Tuple[int, int]:
        ss = max(0, min(int(round(s)), n))
        ee = max(0, min(int(round(e)), n))
        if ee <= ss:
            raise ValueError("Empty window.")
        return ss, ee

    def windows_ok(anchor: int) -> bool:
        try:
            clamp_win(anchor - (110 + 45), anchor - (110 + 35))
            clamp_win(anchor - (110 / 2 + 15), anchor - (110 / 2 + 5))
            clamp_win(anchor - 120, anchor - 95)
            return True
        except Exception:
            return False

    anchor = int(lo + np.argmax(smooth[lo : hi + 1]))
    anchor_mode = "between_blue_red"

    if not windows_ok(anchor):
        anchor = int(np.argmax(smooth))
        anchor_mode = "global_peak"

    if not windows_ok(anchor):
        raise ValueError(
            f"Empty window: anchor={anchor} (mode={anchor_mode}), n={n}. "
            "Pick blue/red closer to the true peak region, or adjust window offsets."
        )

    b1s, b1e = clamp_win(anchor - (110 + 45), anchor - (110 + 35))
    b2s, b2e = clamp_win(anchor - (110 / 2 + 15), anchor - (110 / 2 + 5))
    sgs, sge = clamp_win(anchor - 120, anchor - 95)

    base_l = float(np.mean(smooth[b1s:b1e]))
    base_r = float(np.mean(smooth[b2s:b2e]))
    baseline = float(np.mean([base_l, base_r]))
    signal = float(np.max(smooth[sgs:sge]))
    sbr = float(signal / baseline) if baseline != 0 else float("inf")

    return {
        "ok": True,
        "profile_len": n,
        "x_blue": int(x_blue),
        "x_red": int(x_red),
        "anchor_mode": anchor_mode,
        "anchor_range": {"lo": int(lo), "hi": int(hi)},
        "anchor_peak_x": int(anchor),
        "windows": {
            "baseline_1": {"start": int(b1s), "end": int(b1e), "mean": float(base_l)},
            "baseline_2": {"start": int(b2s), "end": int(b2e), "mean": float(base_r)},
            "signal": {"start": int(sgs), "end": int(sge), "max": float(signal)},
        },
        "baseline_value": float(baseline),
        "signal_value": float(signal),
        "sbr": float(sbr),
        "equations": {
            "baseline_symbolic": "baseline = mean( mean(w1), mean(w2) )",
            "baseline_substitute": f"baseline = mean({_fmt6(base_l)}, {_fmt6(base_r)}) = {_fmt6(baseline)}",
            "signal_symbolic": "signal = max(signal_window)",
            "signal_substitute": f"signal = {_fmt6(signal)}",
            "sbr_symbolic": "SBR = signal / baseline",
            "sbr_substitute": f"SBR = {_fmt6(signal)} / {_fmt6(baseline)}",
            "sbr_result": f"SBR = {_fmt6(sbr)}",
        },
    }


# =========================
# Disk ROI loader hook (set by cell7)
# =========================
_ROI_LOADER: Optional[Callable[[str, str, str], Optional[np.ndarray]]] = None


def set_roi_loader(fn: Callable[[str, str, str], Optional[np.ndarray]]) -> None:
    global _ROI_LOADER
    _ROI_LOADER = fn


def _load_roi_for_walkthrough(date: str, ds_rep_raw: str, row: str) -> Optional[np.ndarray]:
    if _ROI_LOADER is None:
        return None
    return _ROI_LOADER(date, ds_rep_raw, row)


# =========================
# CSV helpers (used by combined graphs + walkthrough)
# =========================
def _load_export_csv_for_dataset(date: str, ds_raw: str) -> Optional[pd.DataFrame]:
    try:
        ds = sanitize_name(ds_raw)
        path = EXPORT_ROOT / date / f"{ds}.csv"
        if not path.exists():
            return None
        df = pd.read_csv(path)
        if "row" not in df.columns:
            return None
        df["row"] = df["row"].astype(str).map(_normalize_row_label)
        return df.set_index("row", drop=True)
    except Exception:
        return None


def _resolve_triplet_datasets(date: str, selected_ds_raw: str) -> Dict[str, Optional[str]]:
    want_lossless = "losslessformat" in (selected_ds_raw or "").lower()

    candidates: List[str] = []
    for ds in DATASETS.keys():
        is_lossless = "losslessformat" in ds.lower()
        if want_lossless != is_lossless:
            continue
        rep = _replicate_from_dataset_name(ds)
        if rep in {"N1", "N2", "N3"}:
            candidates.append(ds)

    base = EXPORT_ROOT / date

    def has_export(ds_name: str) -> bool:
        return (base / f"{sanitize_name(ds_name)}.csv").exists()

    triplet: Dict[str, Optional[str]] = {"N1": None, "N2": None, "N3": None}

    for rep in ["N1", "N2", "N3"]:
        rep_cands = [ds for ds in candidates if _replicate_from_dataset_name(ds) == rep]
        rep_exported = [ds for ds in rep_cands if has_export(ds)]

        if selected_ds_raw in rep_exported:
            triplet[rep] = selected_ds_raw
        elif rep_exported:
            triplet[rep] = sorted(rep_exported)[0]
        elif selected_ds_raw in rep_cands:
            triplet[rep] = selected_ds_raw
        elif rep_cands:
            triplet[rep] = sorted(rep_cands)[0]

    return triplet


# =========================
# Walkthrough/table (cc/k/dip excluded via row collectors)
# =========================
def _walkthrough_for_one_replicate_row(date: str, ds_rep_raw: str, row: str) -> Dict[str, object]:
    df = _load_export_csv_for_dataset(date, ds_rep_raw)
    r = _normalize_row_label(row)
    rep = _replicate_from_dataset_name(ds_rep_raw) or ""

    if df is None or df.empty or r not in df.index:
        return {"ok": False, "dataset": ds_rep_raw, "replicate": rep, "row": r, "msg": "CSV missing row."}

    def get_float(col: str) -> Optional[float]:
        try:
            v = df.loc[r].get(col, "")
            if v in ("", None):
                return None
            f = float(v)
            return f if np.isfinite(f) else None
        except Exception:
            return None

    def get_int(col: str) -> Optional[int]:
        try:
            v = df.loc[r].get(col, "")
            if v in ("", None):
                return None
            return int(float(v))
        except Exception:
            return None

    peak_y = get_float("peak_y")
    baseline_at_peak = get_float("baseline_at_peak")
    if peak_y is not None and baseline_at_peak is not None:
        sbr = float(peak_y / baseline_at_peak) if baseline_at_peak != 0 else float("inf")
        return {
            "ok": True,
            "mode": "saved",
            "dataset": ds_rep_raw,
            "replicate": rep,
            "row": r,
            "signal_value": float(peak_y),
            "baseline_value": float(baseline_at_peak),
            "sbr": float(sbr),
            "equations": {
                "symbolic": "SBR = peak_y / baseline_at_peak",
                "substitute": f"SBR = {_fmt6(peak_y)} / {_fmt6(baseline_at_peak)}",
                "result": f"SBR = {_fmt6(sbr)}",
            },
        }

    x_blue = get_int("x_blue")
    x_red = get_int("x_red")
    if x_blue is None or x_red is None:
        return {
            "ok": False,
            "mode": "missing",
            "dataset": ds_rep_raw,
            "replicate": rep,
            "row": r,
            "msg": "Missing x_blue/x_red and missing saved peak/baseline.",
        }

    roi = _load_roi_for_walkthrough(date=date, ds_rep_raw=ds_rep_raw, row=r)
    if roi is None:
        return {
            "ok": False,
            "mode": "missing_roi",
            "dataset": ds_rep_raw,
            "replicate": rep,
            "row": r,
            "msg": "ROI missing on disk for this replicate/row.",
        }

    details = ui_baseline_details_from_roi(roi, x_blue=x_blue, x_red=x_red, crop_like_notebook=True)
    details.update({"dataset": ds_rep_raw, "replicate": rep, "row": r, "mode": "recomputed"})
    details["signal_value"] = float(details["peak_y"])
    details["baseline_value"] = float(details["baseline_at_peak"])
    return details


def _walkthrough_triplet_for_row(date: str, family_ds_raw: str, row: str) -> List[Dict[str, object]]:
    triplet = _resolve_triplet_datasets(date=date, selected_ds_raw=family_ds_raw)
    out: List[Dict[str, object]] = []
    for rep in ["N1", "N2", "N3"]:
        ds_rep = triplet.get(rep)
        if not ds_rep:
            out.append({"ok": False, "replicate": rep, "dataset": "", "row": _normalize_row_label(row), "msg": "Missing replicate dataset."})
            continue
        out.append(_walkthrough_for_one_replicate_row(date=date, ds_rep_raw=ds_rep, row=row))
    return out


def _walkthrough_all_rows_table(
    date: str,
    family_ds_raw: str,
    *,
    include_dip: bool = False,  # kept for signature compatibility; we always exclude dip here
) -> Dict[str, Any]:
    triplet = _resolve_triplet_datasets(date=date, selected_ds_raw=family_ds_raw)

    EXCLUDE = {"cc", "k", "dip", "dips"}

    row_set: set = set()
    for rep in ["N1", "N2", "N3"]:
        ds_rep = triplet.get(rep)
        if not ds_rep:
            continue
        df = _load_export_csv_for_dataset(date, ds_rep)
        if df is None or df.empty:
            continue
        for r in df.index.tolist():
            rr = _normalize_row_label(str(r))
            if rr in EXCLUDE:
                continue
            row_set.add(rr)

    rows = [r for r in GRAPH_ROW_ORDER if r in row_set]
    rows.extend(sorted([r for r in row_set if r not in rows]))

    cells: Dict[str, Dict[str, Dict[str, object]]] = {}
    for r in rows:
        cells[r] = {}
        blocks = _walkthrough_triplet_for_row(date=date, family_ds_raw=family_ds_raw, row=r)
        for blk in blocks:
            rep = (blk.get("replicate") or "").strip().upper()
            if rep not in {"N1", "N2", "N3"}:
                rep = _replicate_from_dataset_name(str(blk.get("dataset") or "")) or ""
            if rep not in {"N1", "N2", "N3"}:
                continue
            cells[r][rep] = blk

    return {"rows": rows, "replicates": ["N1", "N2", "N3"], "triplet": triplet, "cells": cells}


def _walkthrough_flat_list(
    date: str,
    family_ds_raw: str,
    *,
    include_dip: bool = False,
) -> List[Dict[str, object]]:
    table = _walkthrough_all_rows_table(date=date, family_ds_raw=family_ds_raw, include_dip=include_dip)

    out: List[Dict[str, object]] = []
    for r in table["rows"]:
        for rep in table["replicates"]:
            cell = table["cells"].get(r, {}).get(rep)
            if cell is None:
                out.append({"ok": False, "row": r, "replicate": rep, "msg": "Missing cell."})
            else:
                out.append(cell)
    return out


# =========================
# Combined graph + negativity threshold
# =========================
def _png_message(msg: str) -> bytes:
    fig, ax = plt.subplots(figsize=(8, 2.2))
    ax.axis("off")
    ax.text(0.01, 0.5, msg, va="center", ha="left", fontsize=11)
    buf = io.BytesIO()
    fig.tight_layout()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return buf.getvalue()


def _collect_combined_graph_points(
    date: str, family_ds_raw: str
) -> Tuple[List[float], List[float], List[float], List[str], Optional[float], str]:
    triplet = _resolve_triplet_datasets(date=date, selected_ds_raw=family_ds_raw)

    dfs: Dict[str, pd.DataFrame] = {}
    for rep, ds_raw in triplet.items():
        if not ds_raw:
            continue
        df = _load_export_csv_for_dataset(date, ds_raw)
        if df is None or df.empty:
            continue
        dfs[rep] = df

    if not dfs:
        return [], [], [], [], None, "threshold: n/a"

    row_set: set = set()
    for df in dfs.values():
        for r in df.index.tolist():
            rr = _normalize_row_label(str(r))
            if rr in EXCLUDE_ROWS_GRAPH_FINAL:
                continue
            row_set.add(rr)

    rows = [r for r in GRAPH_ROW_ORDER if r in row_set]
    rows.extend(sorted([r for r in row_set if r not in rows]))

    mat: List[List[float]] = []
    for r in rows:
        vals: List[float] = []
        for rep in ["N1", "N2", "N3"]:
            df = dfs.get(rep)
            if df is None:
                vals.append(float("nan"))
                continue
            try:
                v = df.loc[r].get("row_ratio_mean", float("nan"))
            except Exception:
                v = float("nan")
            try:
                vals.append(float(v))
            except Exception:
                vals.append(float("nan"))
        mat.append(vals)

    arr = np.array(mat, dtype=float)

    # mean across available reps
    avg = np.nanmean(arr, axis=1)

    # SEM across available reps: std(ddof=1)/sqrt(n_valid)
    n_valid = np.sum(np.isfinite(arr), axis=1).astype(float)
    stnd = np.nanstd(arr, axis=1, ddof=1)
    sem = np.where(n_valid > 1, stnd / np.sqrt(n_valid), 0.0)

    thr = None
    thr_meta = "threshold: n/a"
    try:
        if "neg" in rows:
            i = rows.index("neg")
            neg_vals = arr[i, :]
            neg_vals = neg_vals[np.isfinite(neg_vals)]
            if neg_vals.size >= 2:
                mu = float(np.mean(neg_vals))
                sigma = float(np.std(neg_vals, ddof=1))
                thr = float(mu + 3.0 * sigma)
                thr_meta = f"thr = mean_neg + 3*std_neg = {mu:.4f} + 3*{sigma:.4f} = {thr:.4f} (n={neg_vals.size})"
            elif neg_vals.size == 1:
                thr = float(neg_vals[0])
                thr_meta = f"thr = mean_neg (only 1 rep) = {thr:.4f}"
    except Exception:
        thr = None
        thr_meta = "threshold: n/a"

    loads: List[float] = []
    means: List[float] = []
    errs: List[float] = []
    labels: List[str] = []

    for r, a, e in zip(rows, avg, sem):
        bl = _row_to_bacterial_load(r)
        if bl is None or (not np.isfinite(a)):
            continue
        loads.append(float(bl))
        means.append(float(a))
        errs.append(float(e) if np.isfinite(e) else 0.0)
        labels.append(r)

    order = np.argsort(np.array(loads, dtype=float)).tolist()
    loads = [loads[i] for i in order]
    means = [means[i] for i in order]
    errs = [errs[i] for i in order]
    labels = [labels[i] for i in order]

    return loads, means, errs, labels, thr, thr_meta


def _final_graph_combined_png_bytes(date: str, family_ds_raw: str) -> bytes:
    loads, means, errs, labels, thr, thr_meta = _collect_combined_graph_points(date=date, family_ds_raw=family_ds_raw)
    if not loads:
        return _png_message("Combined graph: no plotted points.\nRun update_medians for N1/N2/N3 first.")

    # split neg (x<=0) vs rest
    left_x, left_y, left_e = [], [], []
    right_x, right_y, right_e = [], [], []

    for bl, m, e in zip(loads, means, errs):
        if float(bl) <= 0.0:
            left_x.append(float(bl)); left_y.append(float(m)); left_e.append(float(e))
        else:
            right_x.append(float(bl)); right_y.append(float(m)); right_e.append(float(e))

    # Make sure nothing clips: include top of error bars and threshold
    top_vals = []
    for m, e in zip(means, errs):
        if np.isfinite(m) and np.isfinite(e):
            top_vals.append(float(m + e))
        elif np.isfinite(m):
            top_vals.append(float(m))
    lim = (max(top_vals) + 0.15) if top_vals else 1.2
    lim = max(1.2, lim)
    if thr is not None and np.isfinite(thr):
        lim = max(lim, float(thr) + 0.15)

    f, axes = plt.subplots(1, 2, gridspec_kw={"width_ratios": [1, 6]}, figsize=(9.6, 3.8))
    ax0, ax1 = axes[0], axes[1]
    ax1.yaxis.tick_right()
    ax1.set_xscale("log")

    ax1.set_ylim(0.99, lim)
    ax0.set_ylim(0.99, lim)
    ax0.set_xlim(-0.9, 1)

    # Restrict log-x like the example (avoid showing 10^0..10^4)
    ax1.set_xlim(0.5 * 10**5, 2 * 10**9)

    # Grid like example
    ax0.yaxis.grid(which="major", color="#DDDDDD", zorder=0, linewidth=0.8)
    ax0.xaxis.grid(which="major", color="#DDDDDD", zorder=0, linewidth=0.8)
    ax1.grid(which="major", color="#DDDDDD", zorder=0, linewidth=0.8)
    ax1.grid(which="minor", color="#EEEEEE", zorder=0, linewidth=0.5, linestyle=":")
    ax1.minorticks_on()

    ax0.set_xticks([0])
    ax0.set_xticklabels(["neg"])
    ax0.set_ylabel("Signal to baseline ratio", fontsize=14)
    ax1.set_xlabel("S. pyogenes concentration in saliva [CFU/mL]", fontsize=14)

    # --- Styling like example (open markers + edgecolor) ---
    color_manual = "deepskyblue"
    s = 40

    # NEG point shown in left panel (open circle) + SEM error bar
    if left_x:
        ax0.scatter(left_x, left_y, facecolors="white", edgecolors=color_manual, linewidths=1, s=s, zorder=3, marker="o")
        ax0.scatter(left_x, left_y, c="dimgray", marker="_", s=300, zorder=2)
        ax0.errorbar(left_x, left_y, yerr=left_e, fmt="none", ecolor=color_manual, capsize=4, zorder=3)

    # Non-neg points shown in right panel (open circle) + SEM error bars
    if right_x:
        ax1.scatter(
            right_x, right_y,
            facecolors="white", edgecolors=color_manual, linewidths=1,
            s=s, zorder=3, marker="o",
            label="Manual protocol",
        )
        ax1.errorbar(right_x, right_y, yerr=right_e, fmt="none", ecolor=color_manual, capsize=4, zorder=3)

    # Positivity threshold (dashed)
    if thr is not None and np.isfinite(thr):
        ax1.plot(
            [min(right_x) if right_x else 1e5, max(right_x) if right_x else 1e9],
            [thr, thr],
            linestyle="dashed",
            c="k",
            alpha=0.5,
            label="Positivity threshold",
        )
        ax0.plot([-1, 1], [thr, thr], linestyle="dashed", c="k", alpha=0.5)

        # threshold text in the plot (top-left of right panel)
        ax1.text(
            0.02,
            0.98,
            thr_meta,
            transform=ax1.transAxes,
            va="top",
            ha="left",
            fontsize=9,
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
        )

    # Add legend-only entry "Error Bars are SEM"
    ax1.scatter(
        [right_x[0] if right_x else 1e5],
        [0.0],
        c="none",
        marker="_",
        s=200,
        alpha=0.0,
        label="Error Bars are SEM",
    )

    # Title
    f.suptitle(f"{family_ds_raw} (combined N1/N2/N3)", fontsize=12, y=0.98)

    # ---- NEW: legend ABOVE axes, BELOW title (no overlap with data) ----
    handles, labels_leg = ax1.get_legend_handles_labels()
    f.legend(
        handles,
        labels_leg,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.915),  # between title and axes
        ncol=3,                       # adjust if you have more/less entries
        frameon=True,
        facecolor="white",
        framealpha=0.95,
    )

    # Reduce overlap + make room at top for title+legend
    plt.subplots_adjust(wspace=0.05, hspace=0, top=0.82)

    buf = io.BytesIO()
    f.savefig(buf, format="png", dpi=200, bbox_inches="tight")
    plt.close(f)
    buf.seek(0)
    return buf.getvalue()



# =========================
# Experiments helper (unchanged)
# =========================
def analyze_experiments(R_list: List[np.ndarray]) -> Dict[str, object]:
    if not R_list:
        return {
            "experiment_means": np.array([]),
            "experiment_SEMs": np.array([]),
            "experiment_ns": np.array([]),
            "R_best": float("nan"),
            "SE_best": float("nan"),
            "N_total": 0,
            "R_all": np.array([]),
        }

    exp_means = np.array([np.mean(R) for R in R_list], dtype=float)
    exp_sds = np.array([np.std(R, ddof=1) for R in R_list], dtype=float)
    exp_ns = np.array([int(R.size) for R in R_list], dtype=int)
    exp_sems = exp_sds / np.sqrt(exp_ns.astype(float))

    R_all = np.concatenate(R_list).astype(float)
    N = int(R_all.size)
    R_best = float(np.mean(R_all))
    s_pooled = float(np.std(R_all, ddof=1))
    SE_best = float(s_pooled / np.sqrt(float(N))) if N else float("nan")

    return {
        "experiment_means": exp_means,
        "experiment_SEMs": exp_sems,
        "experiment_ns": exp_ns,
        "R_best": R_best,
        "SE_best": SE_best,
        "N_total": N,
        "R_all": R_all,
    }


In [ ]:
# /content/uwlfa_gui_cells/cell7_routes.py

import json
import io
import math
from pathlib import Path
from typing import Optional, Any, List, Tuple

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from flask import jsonify, request, Response


def _as_path(p: Any) -> Optional[Path]:
    if p is None:
        return None
    if isinstance(p, Path):
        return p
    try:
        return Path(str(p))
    except Exception:
        return None


def _json_error(msg: str, status: int = 500):
    return Response(json.dumps({"ok": False, "msg": msg}), status=status, mimetype="application/json")


# ============================================================
# NEW: exclude rows everywhere in the IMAGE BROWSER dropdown
# ============================================================
EXCLUDE_UI_ROWS = {"cc", "k", "dip", "dips"}  # NEW


def _file_row_norm(f: dict) -> str:  # NEW
    """
    Robustly determine the row label for a DATASETS file dict.
    Prefers the stored 'row' field, falls back to filename parsing.
    """
    try:
        r = f.get("row", "") or ""
        r = _normalize_row_label(str(r))
        if r:
            return r
    except Exception:
        pass
    try:
        fn = f.get("filename", "") or ""
        r = row_name_from_filename(fn)
        return _normalize_row_label(str(r))
    except Exception:
        return ""


def _filtered_files_for_dataset(ds: str) -> List[dict]:  # NEW
    """
    Returns a filtered list of files for ds that EXCLUDES cc/k/dip/dips,
    but preserves internal ordering.

    IMPORTANT: /images, /image, /row_name must all use this so that the
    dropdown index matches what /image loads.
    """
    files = DATASETS.get(ds, []) or []
    out = []
    for f in files:
        r = _file_row_norm(f)
        if r in EXCLUDE_UI_ROWS:
            continue
        out.append(f)
    return out


# ============================================================
# UI-required utility endpoints
# ============================================================

@app.route("/export_info")
def export_info():
    base = (EXPORT_ROOT / DATE_FOLDER) if DATE_FOLDER else EXPORT_ROOT
    return jsonify({"ok": True, "root": str(EXPORT_ROOT), "date": str(DATE_FOLDER), "full": str(base)})


@app.route("/row_name")
def row_name():
    ds = request.args.get("dataset", "")
    idx = int(request.args.get("index", "0"))

    # CHANGED: use filtered list so cc/k/dip don't exist in the UI index-space
    files = _filtered_files_for_dataset(ds)
    if not (0 <= idx < len(files)):
        return jsonify({"ok": False, "row": "", "msg": "Bad index."})
    return jsonify({"ok": True, "row": _file_row_norm(files[idx])})



@app.route("/analyze_grid")
def analyze_grid():
    try:
        date = (request.args.get("date", "") or "").strip() or DATE_FOLDER
        ds_raw = (request.args.get("dataset", "") or "").strip()

        if not date:
            return Response(_png_message("ROI grid error: missing date."), mimetype="image/png")
        if not ds_raw:
            return Response(_png_message("ROI grid error: missing dataset."), mimetype="image/png")

        base = EXPORT_ROOT / date
        if not base.exists():
            return Response(
                _png_message(f"ROI grid error: date folder not found:\n{base.as_posix()}"),
                mimetype="image/png",
            )

        # ALWAYS exclude these rows in the grid
        EXCLUDE = {"cc", "k", "dip", "dips"}

        # ------------------------------------------------------------
        # 1) Prefer the dataset's own file list as the "expected rows"
        # ------------------------------------------------------------
        rows: List[str] = []
        if ds_raw in DATASETS:
            try:
                rows = _dataset_rows_from_files(ds_raw)
                rows = [_normalize_row_label(str(r)) for r in rows]
                rows = [r for r in rows if r and r not in EXCLUDE]
                rows = sorted(set(rows), key=row_sort_key)
            except Exception:
                rows = []

        ds = sanitize_name(ds_raw)
        csv_path = base / f"{ds}.csv"

        # ------------------------------------------------------------
        # 2) Fallback: CSV rows
        # ------------------------------------------------------------
        if not rows and csv_path.exists():
            try:
                df = pd.read_csv(csv_path)
                if "row" in df.columns:
                    rows = [_normalize_row_label(str(x)) for x in df["row"].tolist()]
                    rows = [r for r in rows if r and r not in EXCLUDE]
                    rows = sorted(set(rows), key=row_sort_key)
            except Exception:
                rows = []

        # ------------------------------------------------------------
        # 3) Fallback: scan ROI files on disk
        # ------------------------------------------------------------
        if not rows:
            folder = EXPORT_ROOT / date / ds
            if folder.exists():
                row_set = set()
                for p in folder.glob("ROI_*_*.tif*"):
                    stem = p.stem
                    if not stem.startswith("ROI_"):
                        continue
                    rest = stem[4:]
                    if "_" not in rest:
                        continue
                    row_part = rest.rsplit("_", 1)[0]
                    r = _normalize_row_label(row_part)
                    if r and r not in EXCLUDE:
                        row_set.add(r)
                rows = sorted(row_set, key=row_sort_key)

        if not rows:
            folder = (EXPORT_ROOT / date / ds).as_posix()
            return Response(
                _png_message(
                    "ROI grid: no rows found.\n\n"
                    f"Expected ROI files under:\n{folder}\n\n"
                    "Save ROI for this date/dataset first."
                ),
                mimetype="image/png",
            )

        def load_first_roi(row_label: str):
            p = _first_match(EXPORT_ROOT / date, ds, row_label)
            if not p:
                return None
            return cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)

        n = len(rows)
        cols = 4 if n >= 4 else max(1, n)
        rows_n = int(np.ceil(n / cols))

        fig_w = 9.6
        fig_h = max(2.6, rows_n * 2.2)
        fig, axes = plt.subplots(rows_n, cols, figsize=(fig_w, fig_h))
        if not isinstance(axes, np.ndarray):
            axes = np.array([[axes]])
        axes = axes.reshape(rows_n, cols)

        for i, r in enumerate(rows):
            ax = axes[i // cols, i % cols]
            img = load_first_roi(r)
            if img is None:
                ax.text(0.5, 0.5, f"{r}\n(missing)", ha="center", va="center", fontsize=10)
                ax.set_axis_off()
                continue
            ax.imshow(img, cmap="gray", vmin=0, vmax=255)
            ax.set_title(r, fontsize=10)
            ax.set_xticks([])
            ax.set_yticks([])

        for j in range(n, rows_n * cols):
            axes[j // cols, j % cols].set_axis_off()

        fig.suptitle(f"ROI Grid — {ds_raw} — {date}", fontsize=12)
        buf = io.BytesIO()
        fig.tight_layout()
        fig.savefig(buf, format="png", dpi=180, bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)
        return Response(buf.getvalue(), mimetype="image/png")

    except Exception as e:
        return Response(_png_message(f"ROI grid crashed:\n{type(e).__name__}: {e}"), mimetype="image/png")


# ============================================================
# Core UI routes
# ============================================================

@app.route("/")
def home():
    payload_json = json.dumps(payload)
    html = TEMPLATE_HTML.replace("window.PAYLOAD = __PAYLOAD__;", f"window.PAYLOAD = {payload_json};")
    return Response(html, mimetype="text/html")


@app.route("/datasets")
def datasets():
    return jsonify({"names": list(DATASETS.keys())})

@app.route("/images")
def images():
    ds = request.args.get("dataset", "")
    return jsonify({"files": _filtered_files_for_dataset(ds)})


@app.route("/image")
def image():
    global CURRENT_IMAGE, ORIGINAL_IMAGE

    ds = request.args.get("dataset", "")
    idx = int(request.args.get("index", "0"))

    # CHANGED: index is into the FILTERED list (so it matches the dropdown)
    files = _filtered_files_for_dataset(ds)
    if not (0 <= idx < len(files)):
        img = np.zeros((40, 120, 3), np.uint8)
        ok, buf = cv2.imencode(".png", img)
        return Response(buf.tobytes() if ok else _png_message("Bad index."), mimetype="image/png")

    # load by the actual path in the file dict
    path = files[idx].get("path", "")
    key = (ds, idx, path)

    if getattr(app, "_current_key", None) != key:
        path = files[idx].get("path", None)
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED) if path else None
        if img is None:
            img = np.zeros((40, 120, 3), np.uint8)
        ORIGINAL_IMAGE = img.copy()
        CURRENT_IMAGE = img.copy()
        app._current_key = key

    if CURRENT_IMAGE is None:
        return Response(_png_message("No image loaded."), mimetype="image/png")

    if CURRENT_IMAGE.ndim == 2:
        gray = _to_gray_like_notebook(CURRENT_IMAGE)
        gray01 = _normalize_gray_to_01(gray)
        if DISPLAY_INVERT_INTENSITY:
            gray01 = 1.0 - gray01
        gray_u8 = _autocontrast_u8(gray01, p_lo=1.0, p_hi=99.0)
        rgb = cv2.cvtColor(gray_u8, cv2.COLOR_GRAY2RGB)
        ok, buf = cv2.imencode(".png", rgb)
        if not ok:
            return Response(_png_message("PNG encode failed."), mimetype="image/png")
        return Response(buf.tobytes(), mimetype="image/png")

    img = CURRENT_IMAGE.astype(np.float32)
    if float(np.nanmax(img)) > 1.5:
        denom = _dtype_max_value(CURRENT_IMAGE)
        denom = denom if denom > 0 else (float(np.nanmax(img)) if float(np.nanmax(img)) > 0 else 1.0)
        img = img / float(denom)
    img = np.clip(img, 0.0, 1.0)

    if DISPLAY_INVERT_INTENSITY:
        img = 1.0 - img

    img_u8 = (img * 255.0).clip(0, 255).astype(np.uint8)
    rgb_u8 = cv2.cvtColor(img_u8, cv2.COLOR_BGR2RGB)

    ok, buf = cv2.imencode(".png", rgb_u8)
    if not ok:
        return Response(_png_message("PNG encode failed."), mimetype="image/png")
    return Response(buf.tobytes(), mimetype="image/png")


@app.route("/reset_view", methods=["POST"])
def reset_view():
    global CURRENT_IMAGE, ORIGINAL_IMAGE
    if ORIGINAL_IMAGE is not None:
        CURRENT_IMAGE = ORIGINAL_IMAGE.copy()
        return jsonify({"ok": True, "msg": "View reset."})
    return jsonify({"ok": False, "msg": "No image to reset."})


@app.route("/rotate", methods=["POST"])
def rotate():
    global CURRENT_IMAGE, ORIGINAL_IMAGE
    if CURRENT_IMAGE is None or ORIGINAL_IMAGE is None:
        return jsonify({"ok": False, "msg": "No image loaded."})
    CURRENT_IMAGE = cv2.rotate(CURRENT_IMAGE, cv2.ROTATE_90_COUNTERCLOCKWISE)
    ORIGINAL_IMAGE = cv2.rotate(ORIGINAL_IMAGE, cv2.ROTATE_90_COUNTERCLOCKWISE)
    return jsonify({"ok": True, "msg": "Rotated 90° CCW (persistent)."})


@app.route("/apply_grayscale", methods=["POST"])
def apply_grayscale():
    global CURRENT_IMAGE
    if CURRENT_IMAGE is None:
        return jsonify({"ok": False, "msg": "No image loaded."})
    try:
        CURRENT_IMAGE = _apply_grayscale_preserve_dtype(CURRENT_IMAGE)
    except Exception as e:
        return jsonify({"ok": False, "msg": f"Apply grayscale failed: {type(e).__name__}: {e}"})
    return jsonify({"ok": True, "msg": "Applied grayscale to current view (persistent until Reset View)."})


@app.route("/apply_crop_zoom", methods=["POST"])
def apply_crop_zoom():
    global CURRENT_IMAGE, ORIGINAL_IMAGE
    if CURRENT_IMAGE is None or ORIGINAL_IMAGE is None:
        return jsonify({"ok": False, "msg": "No image loaded."})

    d = request.get_json(force=True)
    nx = float(d.get("nx", 0.0))
    ny = float(d.get("ny", 0.0))
    nw = float(d.get("nw", 0.0))
    nh = float(d.get("nh", 0.0))
    tW = int(d.get("tw", ORIGINAL_IMAGE.shape[1]))
    tH = int(d.get("th", ORIGINAL_IMAGE.shape[0]))

    H, W = CURRENT_IMAGE.shape[:2]
    x1 = int(max(0, min(W - 1, round(nx * W))))
    y1 = int(max(0, min(H - 1, round(ny * H))))
    x2 = int(max(0, min(W, round((nx + nw) * W))))
    y2 = int(max(0, min(H, round((ny + nh) * H))))
    if x2 <= x1 or y2 <= y1:
        return jsonify({"ok": False, "msg": "Empty selection."})

    crop = CURRENT_IMAGE[y1:y2, x1:x2]
    upscale = (crop.shape[1] < tW) or (crop.shape[0] < tH)
    interp = cv2.INTER_CUBIC if upscale else cv2.INTER_AREA
    CURRENT_IMAGE = cv2.resize(crop, (tW, tH), interpolation=interp)
    return jsonify({"ok": True, "msg": f"Cropped [{x1}:{x2}]×[{y1}:{y2}] → {tW}×{tH}."})


@app.route("/save_roi", methods=["POST"])
def save_roi():
    data = request.get_json(force=True)
    ds_raw = data.get("dataset", "")
    idx = int(data.get("index", 0))

    if ds_raw not in DATASETS:
        return jsonify({"ok": False, "msg": f"Unknown dataset: {ds_raw}"})

    # CHANGED: index is filtered-list index, consistent with /images + /image
    files = _filtered_files_for_dataset(ds_raw)
    if not (0 <= idx < len(files)):
        return jsonify({"ok": False, "msg": "Bad image index."})

    ds = sanitize_name(ds_raw)
    row = sanitize_name(row_name_from_filename(files[idx]["filename"]))
    row = _normalize_row_label(row)

    # hard guard even though filtered list should prevent this
    if row in EXCLUDE_UI_ROWS:
        return jsonify({"ok": False, "msg": f"Refusing to save ROI for excluded row '{row}'."})

    try:
        roi = _normalized_roi_from_current()
    except Exception as e:
        return jsonify({"ok": False, "msg": f"ROI not available: {e}"})

    out_date_dir = EXPORT_ROOT / DATE_FOLDER
    out_ds_dir = out_date_dir / ds
    out_ds_dir.mkdir(parents=True, exist_ok=True)

    key = (ds, row)
    ROI_SAVE_COUNT[key] = ROI_SAVE_COUNT.get(key, 0) + 1
    n = ROI_SAVE_COUNT[key]

    out_npy = out_ds_dir / f"ROI_{row}_{n}.npy"
    out_tif = out_ds_dir / f"ROI_{row}_{n}.tiff"

    np.save(str(out_npy), roi.astype(np.float32))
    cv2.imwrite(str(out_tif), (roi * 255.0).clip(0, 255).astype(np.uint8))

    ROI_LATEST_PATH[key] = out_npy
    ROI_LATEST_SHAPE[key] = (int(roi.shape[0]), int(roi.shape[1]))

    return jsonify({"ok": True, "msg": f"Saved ROI → {out_npy.name} + {out_tif.name} (dataset={ds_raw}, row={row})"})


@app.route("/export_df", methods=["POST"])
def export_df():
    global DF_GRAY, DF_DATASET, DF_CSV_PATH

    data = request.get_json(force=True) if request.data else {}
    ds_raw = (data.get("dataset") or "").strip()
    if ds_raw not in DATASETS:
        return jsonify({"ok": False, "msg": f"Unknown dataset: {ds_raw}"})

    ds = sanitize_name(ds_raw)
    rows = _dataset_rows_from_files(ds_raw)

    records = []
    for row in rows:
        key = (ds, row)
        roi_path = ROI_LATEST_PATH.get(key)
        shape = ROI_LATEST_SHAPE.get(key)
        records.append(
            {
                "row": row,
                "roi_path": str(roi_path) if roi_path else "",
                "shape": f"{shape[0]}x{shape[1]}" if shape else "0x0",

                "x_blue": "",
                "x_red": "",

                "base_x_1": "",
                "base_x_2": "",
                "peak_x": "",
                "peak_y": "",
                "baseline_at_peak": "",
                "baseline_x1": "",
                "baseline_y1": "",
                "baseline_x2": "",
                "baseline_y2": "",

                "row_ratio_mean": "",
            }
        )

    df = pd.DataFrame.from_records(records).set_index("row")

    out_date_dir = EXPORT_ROOT / DATE_FOLDER
    out_date_dir.mkdir(parents=True, exist_ok=True)
    out_csv = out_date_dir / f"{ds}.csv"
    df.to_csv(out_csv)

    DF_GRAY = df.copy()
    DF_DATASET = ds_raw
    DF_CSV_PATH = out_csv

    return jsonify({"ok": True, "msg": f"Wrote {out_csv} (rows={len(df)})"})


@app.route("/load_df", methods=["POST"])
def load_df():
    global DF_GRAY, DF_DATASET, DF_CSV_PATH

    data = request.get_json(force=True) if request.data else {}
    ds_raw = (data.get("dataset") or "").strip()
    date = (data.get("date") or "").strip() or DATE_FOLDER

    if not ds_raw:
        return jsonify({"ok": False, "msg": "Load DF: missing dataset."})

    if ds_raw not in DATASETS:
        rev = None
        for k in DATASETS.keys():
            if sanitize_name(k) == ds_raw:
                rev = k
                break
        if rev is None:
            return jsonify({"ok": False, "msg": f"Load DF: unknown dataset: {ds_raw}"})
        ds_raw = rev

    ds = sanitize_name(ds_raw)
    base = EXPORT_ROOT / date
    if not base.exists():
        return jsonify({"ok": False, "msg": f"Load DF: date folder not found: {base.as_posix()}"})

    csv_path = base / f"{ds}.csv"
    if not csv_path.exists():
        return jsonify({"ok": False, "msg": f"Load DF: CSV not found: {csv_path.as_posix()}"})

    try:
        df = pd.read_csv(csv_path, index_col="row")
    except Exception as e:
        return jsonify({"ok": False, "msg": f"Load DF: read failed: {type(e).__name__}: {e}"})

    try:
        df.index = df.index.map(lambda x: _normalize_row_label(str(x)))
    except Exception:
        pass

    DF_GRAY = df.copy()
    DF_DATASET = ds_raw
    DF_CSV_PATH = csv_path

    return jsonify({"ok": True, "msg": f"Loaded {csv_path.name} (rows={len(df)})", "dataset": ds_raw, "date": date})


@app.route("/df_rows")
def df_rows():
    if DF_GRAY is None or not isinstance(DF_GRAY, pd.DataFrame) or DF_GRAY.empty:
        return jsonify({"ok": False, "rows": [], "msg": "No exported CSV/DF yet. Click 'Export CSV for Dataset' first."})

    # CHANGED: filter cc/k/dip out of row dropdown too
    rows = []
    for x in DF_GRAY.index.tolist():
        r = _normalize_row_label(str(x))
        if r in EXCLUDE_UI_ROWS:
            continue
        rows.append(str(x))

    return jsonify({"ok": True, "rows": rows, "dataset": DF_DATASET or ""})


@app.route("/exports")
def list_exports():
    dates = []
    if EXPORT_ROOT.exists():
        for p in sorted(EXPORT_ROOT.iterdir()):
            if p.is_dir():
                dates.append(p.name)
    return jsonify({"dates": dates})


# ============================================================
# Replicate-aware ROI loader from disk + hook for cell6
# ============================================================

def _resolve_replicate_dataset_raw(date: str, family_ds_raw: str, replicate: str) -> str:
    replicate = (replicate or "").strip().upper()
    if replicate not in {"N1", "N2", "N3"}:
        replicate = "N1"

    ds_raw = (family_ds_raw or "").strip()

    if ds_raw and ds_raw not in DATASETS:
        rev = None
        for k in DATASETS.keys():
            if sanitize_name(k) == ds_raw:
                rev = k
                break
        if rev is None:
            return ""
        ds_raw = rev

    if not ds_raw:
        return ""

    triplet = _resolve_triplet_datasets(date=date, selected_ds_raw=ds_raw)
    return (triplet.get(replicate) or "").strip()


def _load_exported_roi_for(date: str, ds_rep_raw: str, row: str):
    date = (date or "").strip()
    ds_rep_raw = (ds_rep_raw or "").strip()
    row = _normalize_row_label((row or "").strip())

    if not date:
        return None, "Missing date."
    if not ds_rep_raw:
        return None, "Missing replicate dataset."
    if not row:
        return None, "Missing row."

    base = EXPORT_ROOT / date
    if not base.exists():
        return None, f"Date folder not found: {base.as_posix()}"

    ds_rep = sanitize_name(ds_rep_raw)

    match = _first_match(base, ds_rep, row)
    tif_path = _as_path(match)
    if tif_path is None:
        return None, f"No ROI file found for row '{row}'. Save ROI first."

    npy_path = tif_path.with_suffix(".npy")
    if npy_path.exists():
        try:
            roi = np.load(str(npy_path)).astype(np.float32)
            roi = np.clip(roi, 0.0, 1.0)
            return roi, ""
        except Exception:
            pass

    img = cv2.imread(str(tif_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None, f"Failed to read ROI TIFF: {tif_path.name}"

    roi = (img.astype(np.float32) / 255.0).clip(0.0, 1.0)
    return roi, ""


def _roi_loader_for_cell6(date: str, ds_rep_raw: str, row: str) -> Optional[np.ndarray]:
    roi, err = _load_exported_roi_for(date=date, ds_rep_raw=ds_rep_raw, row=row)
    if err:
        return None
    return roi


set_roi_loader(_roi_loader_for_cell6)


# ============================================================
# Profile endpoint (what the UI plots)
# ============================================================

@app.route("/sbr_profile_json")
def sbr_profile_json():
    try:
        date = (request.args.get("date", "") or "").strip() or DATE_FOLDER
        family_ds = (request.args.get("dataset", "") or "").strip() or (DF_DATASET or "")
        replicate = (request.args.get("replicate", "") or "").strip().upper() or "N1"
        row = (request.args.get("row", "") or "").strip()

        if not row:
            return jsonify({"ok": False, "msg": "Missing row."})

        ds_rep_raw = _resolve_replicate_dataset_raw(date=date, family_ds_raw=family_ds, replicate=replicate)
        if not ds_rep_raw:
            return jsonify({"ok": False, "msg": f"Could not resolve {replicate} dataset."})

        roi, err = _load_exported_roi_for(date=date, ds_rep_raw=ds_rep_raw, row=row)
        if err:
            return jsonify({"ok": False, "msg": err})

        prof = _roi_profile(roi, crop_like_notebook=True)
        return jsonify(
            {
                "ok": True,
                "date": date,
                "dataset": ds_rep_raw,
                "replicate": replicate,
                "row": _normalize_row_label(row),
                "n": int(prof.size),
                "profile": prof.astype(float).tolist(),
            }
        )
    except Exception as e:
        return _json_error(f"sbr_profile_json crashed: {type(e).__name__}: {e}", status=500)


# ============================================================
# CORE: update_medians (baseline-line method)
# ============================================================

@app.route("/update_medians", methods=["POST"])
def update_medians():
    global DF_GRAY, DF_DATASET, DF_CSV_PATH

    try:
        data = request.get_json(force=True) if request.data else {}

        date = (data.get("date") or "").strip() or DATE_FOLDER
        family_ds = (data.get("dataset") or "").strip() or (DF_DATASET or "")
        replicate = (data.get("replicate") or "").strip().upper() or "N1"
        row = (data.get("row") or "").strip()

        try:
            x_blue = int(data.get("x_blue"))
            x_red = int(data.get("x_red"))
        except Exception:
            return jsonify({"ok": False, "msg": "Bad payload. Expected {row, x_blue, x_red}."})

        if not row:
            return jsonify({"ok": False, "msg": "Missing row."})

        ds_rep_raw = _resolve_replicate_dataset_raw(date=date, family_ds_raw=family_ds, replicate=replicate)
        if not ds_rep_raw:
            return jsonify({"ok": False, "msg": f"Could not resolve {replicate} dataset."})

        base = EXPORT_ROOT / date
        ds_rep = sanitize_name(ds_rep_raw)
        csv_path = base / f"{ds_rep}.csv"
        if not csv_path.exists():
            return jsonify({"ok": False, "msg": f"CSV not found for replicate: {csv_path.as_posix()} (Export CSV first)."})

        try:
            df = pd.read_csv(csv_path, index_col="row")
        except Exception as e:
            return jsonify({"ok": False, "msg": f"Read failed: {type(e).__name__}: {e}"})

        try:
            df.index = df.index.map(lambda x: _normalize_row_label(str(x)))
        except Exception:
            pass

        row_norm = _normalize_row_label(row)
        if row_norm not in df.index:
            return jsonify({"ok": False, "msg": f"Row not in CSV: {row_norm}"})

        roi, err = _load_exported_roi_for(date=date, ds_rep_raw=ds_rep_raw, row=row_norm)
        if err:
            return jsonify({"ok": False, "msg": err})

        ui = ui_baseline_details_from_roi(
            roi,
            x_blue=x_blue,
            x_red=x_red,
            crop_like_notebook=True,
        )

        bl = ui["baseline_line"]

        cols_needed = (
            "x_blue","x_red",
            "base_x_1","base_x_2",
            "peak_x","peak_y",
            "baseline_at_peak",
            "baseline_x1","baseline_y1","baseline_x2","baseline_y2",
            "row_ratio_mean",
        )
        for col in cols_needed:
            if col not in df.columns:
                df[col] = ""

        df.at[row_norm, "x_blue"] = int(x_blue)
        df.at[row_norm, "x_red"] = int(x_red)

        df.at[row_norm, "base_x_1"] = int(ui["base_x_1"])
        df.at[row_norm, "base_x_2"] = int(ui["base_x_2"])
        df.at[row_norm, "peak_x"] = int(ui["peak_x"])
        df.at[row_norm, "peak_y"] = float(ui["peak_y"])
        df.at[row_norm, "baseline_at_peak"] = float(ui["baseline_at_peak"])

        df.at[row_norm, "baseline_x1"] = int(bl["x1"])
        df.at[row_norm, "baseline_y1"] = float(bl["y1"])
        df.at[row_norm, "baseline_x2"] = int(bl["x2"])
        df.at[row_norm, "baseline_y2"] = float(bl["y2"])

        df.at[row_norm, "row_ratio_mean"] = float(ui["sbr"])

        df.to_csv(csv_path)

        DF_GRAY = df.copy()
        DF_DATASET = ds_rep_raw
        DF_CSV_PATH = csv_path

        return jsonify(
            {
                "ok": True,
                "msg": (
                    f"Updated {csv_path.name} ({replicate}) row={row_norm}: "
                    f"peak={float(ui['peak_y']):.6f}, baseline@peak={float(ui['baseline_at_peak']):.6f}, "
                    f"SBR={float(ui['sbr']):.6f}"
                ),
                "row": row_norm,
                "replicate": replicate,
                "dataset": ds_rep_raw,

                "peak_x": int(ui["peak_x"]),
                "peak_value": float(ui["peak_y"]),
                "baseline_value": float(ui["baseline_at_peak"]),
                "baseline_xhalf": float(ui["peak_x"]),

                "overlay": {
                    "base_x_1": int(ui["base_x_1"]),
                    "base_x_2": int(ui["base_x_2"]),
                    "peak_x": int(ui["peak_x"]),
                    "peak_y": float(ui["peak_y"]),
                    "baseline_at_peak": float(ui["baseline_at_peak"]),
                    "baseline_line": bl,
                },

                "details": ui,
            }
        )

    except Exception as e:
        return _json_error(f"update_medians crashed: {type(e).__name__}: {e}", status=500)


# ============================================================
# Walkthrough + table endpoints (same names your JS uses)
# ============================================================

@app.route("/sbr_walkthrough_json")
def sbr_walkthrough_json():
    try:
        date = (request.args.get("date", "") or "").strip() or DATE_FOLDER
        family_ds = (request.args.get("dataset", "") or "").strip() or (DF_DATASET or "")
        row = (request.args.get("row", "") or "").strip()
        if not family_ds:
            return jsonify({"ok": False, "msg": "Missing dataset."})
        if not row:
            return jsonify({"ok": False, "msg": "Missing row."})

        if family_ds not in DATASETS:
            rev = None
            for k in DATASETS.keys():
                if sanitize_name(k) == family_ds:
                    rev = k
                    break
            if rev is None:
                return jsonify({"ok": False, "msg": f"Unknown dataset: {family_ds}"})
            family_ds = rev

        # NOTE: your cell6 table builder should already be excluding cc/k/dip/dips
        table = _walkthrough_all_rows_table(date=date, family_ds_raw=family_ds, include_dip=True)
        r = _normalize_row_label(row)
        cells = table.get("cells", {}).get(r, {})
        items = []
        for rep in ["N1", "N2", "N3"]:
            blk = cells.get(rep)
            if blk is None:
                items.append({"ok": False, "replicate": rep, "row": r, "msg": "Missing cell."})
            else:
                items.append(blk)

        return jsonify({"ok": True, "date": date, "dataset": family_ds, "row": r, "items": items})

    except Exception as e:
        return _json_error(f"sbr_walkthrough_json crashed: {type(e).__name__}: {e}", status=500)


@app.route("/sbr_table_json")
def sbr_table_json():
    try:
        date = (request.args.get("date", "") or "").strip() or DATE_FOLDER
        family_ds = (request.args.get("dataset", "") or "").strip() or (DF_DATASET or "")
        include_dip = (request.args.get("include_dip", "0") or "0").strip().lower() in {"1", "true", "yes", "y"}

        if not family_ds:
            return jsonify({"ok": False, "msg": "Missing dataset."})

        if family_ds not in DATASETS:
            rev = None
            for k in DATASETS.keys():
                if sanitize_name(k) == family_ds:
                    rev = k
                    break
            if rev is None:
                return jsonify({"ok": False, "msg": f"Unknown dataset: {family_ds}"})
            family_ds = rev

        table = _walkthrough_all_rows_table(date=date, family_ds_raw=family_ds, include_dip=include_dip)
        return jsonify({"ok": True, "date": date, "dataset": family_ds, "table": table})

    except Exception as e:
        return _json_error(f"sbr_table_json crashed: {type(e).__name__}: {e}", status=500)


@app.route("/sbr_walkthrough_flat_json")
def sbr_walkthrough_flat_json():
    try:
        date = (request.args.get("date", "") or "").strip() or DATE_FOLDER
        family_ds = (request.args.get("dataset", "") or "").strip() or (DF_DATASET or "")
        include_dip = (request.args.get("include_dip", "0") or "0").strip().lower() in {"1", "true", "yes", "y"}

        if not family_ds:
            return jsonify({"ok": False, "msg": "Missing dataset."})

        if family_ds not in DATASETS:
            rev = None
            for k in DATASETS.keys():
                if sanitize_name(k) == family_ds:
                    rev = k
                    break
            if rev is None:
                return jsonify({"ok": False, "msg": f"Unknown dataset: {family_ds}"})
            family_ds = rev

        items = _walkthrough_flat_list(date=date, family_ds_raw=family_ds, include_dip=include_dip)
        return jsonify({"ok": True, "date": date, "dataset": family_ds, "items": items})

    except Exception as e:
        return _json_error(f"sbr_walkthrough_flat_json crashed: {type(e).__name__}: {e}", status=500)


# ============================================================
# Demo + Experiments (unchanged)
# ============================================================

EXPERIMENTS: List[dict] = []


def _read_xy_from_csv(date: str, ds_rep_raw: str, row_norm: str) -> Tuple[Optional[int], Optional[int], str]:
    base = EXPORT_ROOT / date
    ds_rep = sanitize_name(ds_rep_raw)
    csv_path = base / f"{ds_rep}.csv"
    if not csv_path.exists():
        return None, None, f"CSV not found: {csv_path.as_posix()}"

    try:
        df = pd.read_csv(csv_path, index_col="row")
        df.index = df.index.map(lambda x: _normalize_row_label(str(x)))
    except Exception as e:
        return None, None, f"CSV read failed: {type(e).__name__}: {e}"

    if row_norm not in df.index:
        return None, None, f"Row not in CSV: {row_norm}"

    def _to_int(v) -> Optional[int]:
        try:
            if v is None or v == "":
                return None
            if isinstance(v, float) and not np.isfinite(v):
                return None
            return int(float(v))
        except Exception:
            return None

    xb = _to_int(df.loc[row_norm].get("x_blue", None))
    xr = _to_int(df.loc[row_norm].get("x_red", None))
    if xb is None or xr is None:
        return xb, xr, "Missing x_blue/x_red in CSV (click blue+red once and save)."
    return xb, xr, ""



@app.route("/demo_strip.png")
def demo_strip_png():
    try:
        date = (request.args.get("date", "") or "").strip() or DATE_FOLDER
        family_ds = (request.args.get("dataset", "") or "").strip() or (DF_DATASET or "")
        row = (request.args.get("row", "") or "").strip()
        replicate = (request.args.get("replicate", "") or "N1").strip().upper()

        if not row:
            return Response(_png_message("demo_strip: missing row"), mimetype="image/png")

        ds_rep_raw = _resolve_replicate_dataset_raw(date=date, family_ds_raw=family_ds, replicate=replicate)
        if not ds_rep_raw:
            return Response(_png_message(f"demo_strip: could not resolve {replicate} dataset"), mimetype="image/png")

        roi, err = _load_exported_roi_for(date=date, ds_rep_raw=ds_rep_raw, row=row)
        if err:
            return Response(_png_message(f"demo_strip: {err}"), mimetype="image/png")

        img = (roi * 255.0).clip(0, 255).astype(np.uint8)
        ok, buf = cv2.imencode(".png", img)
        if not ok:
            return Response(_png_message("demo_strip: PNG encode failed"), mimetype="image/png")
        return Response(buf.tobytes(), mimetype="image/png")

    except Exception as e:
        return Response(_png_message(f"demo_strip crashed:\n{type(e).__name__}: {e}"), mimetype="image/png")


@app.route("/demo_average.png")
def demo_average_png():
    try:
        date = (request.args.get("date", "") or "").strip() or DATE_FOLDER
        family_ds = (request.args.get("dataset", "") or "").strip() or (DF_DATASET or "")
        row = (request.args.get("row", "") or "").strip()
        replicate = (request.args.get("replicate", "") or "N1").strip().upper()

        if not row:
            return Response(_png_message("demo_average: missing row"), mimetype="image/png")

        ds_rep_raw = _resolve_replicate_dataset_raw(date=date, family_ds_raw=family_ds, replicate=replicate)
        if not ds_rep_raw:
            return Response(_png_message(f"demo_average: could not resolve {replicate} dataset"), mimetype="image/png")

        roi, err = _load_exported_roi_for(date=date, ds_rep_raw=ds_rep_raw, row=row)
        if err:
            return Response(_png_message(f"demo_average: {err}"), mimetype="image/png")

        prof = _roi_profile(roi, crop_like_notebook=True)

        fig = plt.figure(figsize=(6.8, 2.6))
        plt.plot(np.arange(prof.size), prof)
        plt.title(f"profile — {replicate} — {_normalize_row_label(row)}")
        plt.xlabel("x")
        plt.ylabel("intensity")
        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=180, bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)
        return Response(buf.getvalue(), mimetype="image/png")

    except Exception as e:
        return Response(_png_message(f"demo_average crashed:\n{type(e).__name__}: {e}"), mimetype="image/png")


@app.route("/demo_full_json")
def demo_full_json():
    try:
        date = (request.args.get("date", "") or "").strip() or DATE_FOLDER
        family_ds = (request.args.get("dataset", "") or "").strip() or (DF_DATASET or "")
        row = (request.args.get("row", "") or "").strip()
        replicate = (request.args.get("replicate", "") or "N1").strip().upper()

        if not row:
            return jsonify({"ok": False, "msg": "demo_full_json: missing row"})

        ds_rep_raw = _resolve_replicate_dataset_raw(date=date, family_ds_raw=family_ds, replicate=replicate)
        if not ds_rep_raw:
            return jsonify({"ok": False, "msg": f"demo_full_json: could not resolve {replicate} dataset"})

        row_norm = _normalize_row_label(row)

        # optional overrides
        xb_raw = request.args.get("x_blue", None)
        xr_raw = request.args.get("x_red", None)

        x_blue = None
        x_red = None
        if xb_raw is not None and xr_raw is not None:
            try:
                x_blue = int(float(xb_raw))
                x_red = int(float(xr_raw))
            except Exception:
                x_blue = None
                x_red = None

        # fallback to CSV saved clicks
        if x_blue is None or x_red is None:
            x_blue, x_red, err = _read_xy_from_csv(date=date, ds_rep_raw=ds_rep_raw, row_norm=row_norm)
            if err:
                return jsonify({"ok": False, "msg": err})

        roi, err = _load_exported_roi_for(date=date, ds_rep_raw=ds_rep_raw, row=row_norm)
        if err:
            return jsonify({"ok": False, "msg": err})

        # same baseline-line math as update_medians
        ui = ui_baseline_details_from_roi(
            roi,
            x_blue=int(x_blue),
            x_red=int(x_red),
            crop_like_notebook=True,
        )

        R = float(ui["sbr"])
        demo = {"row_ratio_mean": R, "row_ratio_sem": 0.0, "row_ratio_n": 1}

        return jsonify(
            {
                "ok": True,
                "date": date,
                "dataset": family_ds,
                "dataset_rep": ds_rep_raw,
                "replicate": replicate,
                "row_name": row_norm,
                "row": row_norm,
                "demo": demo,
                "details": ui,
                "results": {"R_all": [R], "note": "demo_full_json (baseline-line method)"},
            }
        )

    except Exception as e:
        return jsonify({"ok": False, "msg": f"demo_full_json crashed: {type(e).__name__}: {e}"})


@app.route("/experiment_clear", methods=["POST"])
def experiment_clear():
    EXPERIMENTS.clear()
    return jsonify({"ok": True, "msg": "Cleared experiments."})


@app.route("/experiment_add", methods=["POST"])
def experiment_add():
    try:
        data = request.get_json(force=True) if request.data else {}
        date = (data.get("date") or "").strip() or DATE_FOLDER
        family_ds = (data.get("dataset") or "").strip() or (DF_DATASET or "")
        row = (data.get("row") or "").strip()
        replicate = (data.get("replicate") or "N1").strip().upper()

        if not row:
            return jsonify({"ok": False, "msg": "experiment_add: missing row"})

        ds_rep_raw = _resolve_replicate_dataset_raw(date=date, family_ds_raw=family_ds, replicate=replicate)
        if not ds_rep_raw:
            return jsonify({"ok": False, "msg": f"experiment_add: could not resolve {replicate} dataset"})

        row_norm = _normalize_row_label(row)

        # optional x_blue/x_red override
        x_blue = data.get("x_blue", None)
        x_red = data.get("x_red", None)
        try:
            x_blue = int(float(x_blue)) if x_blue is not None else None
            x_red = int(float(x_red)) if x_red is not None else None
        except Exception:
            x_blue = None
            x_red = None

        # fallback to CSV saved clicks
        if x_blue is None or x_red is None:
            x_blue, x_red, err = _read_xy_from_csv(date=date, ds_rep_raw=ds_rep_raw, row_norm=row_norm)
            if err:
                return jsonify({"ok": False, "msg": err})

        roi, err = _load_exported_roi_for(date=date, ds_rep_raw=ds_rep_raw, row=row_norm)
        if err:
            return jsonify({"ok": False, "msg": err})

        ui = ui_baseline_details_from_roi(
            roi,
            x_blue=int(x_blue),
            x_red=int(x_red),
            crop_like_notebook=True,
        )
        R = float(ui["sbr"])
        EXPERIMENTS.append({"row": row_norm, "replicate": replicate, "R": R})

        Rs = [float(x["R"]) for x in EXPERIMENTS if np.isfinite(x.get("R", float("nan")))]
        N = len(Rs)
        mu = float(np.mean(Rs)) if N else float("nan")
        se = 0.0
        if N >= 2:
            se = float(np.std(Rs, ddof=1) / math.sqrt(N))

        return jsonify(
            {
                "ok": True,
                "msg": f"Added experiment: row={row_norm}, {replicate}, R={R:.6f} (total N={N})",
                "R_best": mu,
                "SE_best": se,
                "N_total": N,
            }
        )

    except Exception as e:
        return jsonify({"ok": False, "msg": f"experiment_add crashed: {type(e).__name__}: {e}"})


@app.route("/experiment_plot.png")
def experiment_plot_png():
    if not EXPERIMENTS:
        return Response(_png_message("No experiments yet. Click 'Add as Experiment' first."), mimetype="image/png")

    Rs = [float(x["R"]) for x in EXPERIMENTS if np.isfinite(x.get("R", float("nan")))]
    if not Rs:
        return Response(_png_message("No valid experiment R values."), mimetype="image/png")

    fig = plt.figure(figsize=(6.2, 3.0))
    plt.plot(np.arange(len(Rs)), Rs, marker="o")
    plt.title("Experiments (pooled)")
    plt.xlabel("experiment #")
    plt.ylabel("SBR")
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=180, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return Response(buf.getvalue(), mimetype="image/png")


# ============================================================
# Final combined graph (your existing)
# ============================================================

@app.route("/final_graph_combined.png")
def final_graph_combined_png():
    try:
        date = (request.args.get("date", "") or "").strip() or DATE_FOLDER
        family_ds = (request.args.get("dataset", "") or "").strip() or (DF_DATASET or "")

        if not date:
            return Response(_png_message("Combined graph error: missing date."), mimetype="image/png")
        if not family_ds:
            return Response(_png_message("Combined graph error: missing dataset."), mimetype="image/png")

        if family_ds not in DATASETS:
            rev = None
            for k in DATASETS.keys():
                if sanitize_name(k) == family_ds:
                    rev = k
                    break
            if rev is None:
                return Response(_png_message(f"Unknown dataset:\n{family_ds}"), mimetype="image/png")
            family_ds = rev

        return Response(_final_graph_combined_png_bytes(date=date, family_ds_raw=family_ds), mimetype="image/png")

    except Exception as e:
        return Response(_png_message(f"Combined graph crashed:\n{type(e).__name__}: {e}"), mimetype="image/png")


# ============================================================
# Final single-replicate graph (unchanged fallback)
# ============================================================

def _final_graph_single_replicate_png_bytes(date: str, ds_raw: str) -> bytes:
    df = _load_export_csv_for_dataset(date, ds_raw)
    if df is None or df.empty:
        return _png_message(f"Selected graph: missing/empty CSV for\n{ds_raw}\n(date {date})")

    loads: List[float] = []
    ys: List[float] = []

    for r in df.index.tolist():
        rr = _normalize_row_label(str(r))
        if rr in EXCLUDE_ROWS_GRAPH_FINAL:
            continue
        bl = _row_to_bacterial_load(rr)
        if bl is None:
            continue
        try:
            y = float(df.loc[rr].get("row_ratio_mean", float("nan")))
        except Exception:
            y = float("nan")
        if not np.isfinite(y):
            continue
        loads.append(float(bl))
        ys.append(float(y))

    if not loads:
        return _png_message("Selected graph: no valid plotted points.\nRun update_medians for that replicate first.")

    left_x, left_y = [], []
    right_x, right_y = [], []
    for x, y in zip(loads, ys):
        if x <= 0.0:
            left_x.append(x); left_y.append(y)
        else:
            right_x.append(x); right_y.append(y)

    lim = float(np.nanmax(np.array(ys, dtype=float))) if ys else 1.2
    lim = max(1.2, lim + 0.2)

    f, axes = plt.subplots(1, 2, gridspec_kw={"width_ratios": [1, 6]}, figsize=(9.6, 3.8))
    ax0, ax1 = axes[0], axes[1]
    ax1.yaxis.tick_right()
    ax1.set_xscale("log")

    ax1.set_ylim(0.99, lim)
    ax0.set_ylim(0.99, lim)
    ax0.set_xlim(-0.9, 1)

    ax0.yaxis.grid(which="major", color="#DDDDDD", zorder=0, linewidth=0.8)
    ax0.xaxis.grid(which="major", color="#DDDDDD", zorder=0, linewidth=0.8)
    ax1.grid(which="major", color="#DDDDDD", zorder=0, linewidth=0.8)
    ax1.grid(which="minor", color="#EEEEEE", zorder=0, linewidth=0.5)
    ax1.minorticks_on()

    ax0.set_xticks([0])
    ax0.set_xticklabels(["neg"])
    ax0.set_ylabel("Signal to baseline ratio", fontsize=14)

    if left_x:
        ax0.scatter(left_x, left_y, alpha=1)
    if right_x:
        ax1.scatter(right_x, right_y, alpha=1)

    ax1.set_xlabel("S. pyogenes concentration in saliva [CFU/mL]", fontsize=14)
    f.suptitle(f"{ds_raw} (single replicate)", fontsize=12, y=0.98)

    buf = io.BytesIO()
    plt.tight_layout()
    f.savefig(buf, format="png", dpi=200, bbox_inches="tight")
    plt.close(f)
    buf.seek(0)
    return buf.getvalue()


@app.route("/final_graph_selected.png")
def final_graph_selected_png():
    try:
        date = (request.args.get("date", "") or "").strip() or DATE_FOLDER
        family_ds = (request.args.get("dataset", "") or "").strip() or (DF_DATASET or "")
        mode = (request.args.get("mode", "") or "N1").strip().upper()
        if mode not in {"N1", "N2", "N3"}:
            mode = "N1"

        if not date:
            return Response(_png_message("Selected graph error: missing date."), mimetype="image/png")
        if not family_ds:
            return Response(_png_message("Selected graph error: missing dataset."), mimetype="image/png")

        if family_ds not in DATASETS:
            rev = None
            for k in DATASETS.keys():
                if sanitize_name(k) == family_ds:
                    rev = k
                    break
            if rev is None:
                return Response(_png_message(f"Unknown dataset:\n{family_ds}"), mimetype="image/png")
            family_ds = rev

        ds_rep_raw = _resolve_replicate_dataset_raw(date=date, family_ds_raw=family_ds, replicate=mode)
        if not ds_rep_raw:
            return Response(_png_message(f"Could not resolve dataset for {mode}."), mimetype="image/png")

        png = _final_graph_single_replicate_png_bytes(date=date, ds_raw=ds_rep_raw)
        return Response(png, mimetype="image/png")

    except Exception as e:
        return Response(_png_message(f"Selected graph crashed:\n{type(e).__name__}: {e}"), mimetype="image/png")


In [ ]:
# /content/uwlfa_gui_cells/cell8_run_server.py

from werkzeug.serving import make_server  # noqa


class ServerThread(threading.Thread):
    def __init__(self, flask_app: Flask):
        super().__init__(daemon=True)
        self.srv = make_server("127.0.0.1", 0, flask_app)
        self.port = self.srv.server_port

    def run(self):
        self.srv.serve_forever()


server = ServerThread(app)
server.start()

try:
    from google.colab import output as colab_output  # type: ignore

    proxied = colab_output.eval_js(f"google.colab.kernel.proxyPort({server.port})")
    display(
        HTML(
            f'<a href="{proxied}" target="_blank" '
            f'style="font-weight:700;background:#2a6de0;color:#fff;'
            f'padding:10px 14px;border-radius:8px;text-decoration:none">Open UW-LFA GUI in a new tab</a>'
        )
    )
except Exception:
    print(f"Open: http://127.0.0.1:{server.port}")

print(f"Repo: {REPO_DIR}\nDatabase: {DB_DIR}\n")
print("Loaded folders:")
for k in DATASETS:
    print(f" - {k}: {[d['filename'] for d in DATASETS[k]]}")
print(f"\nSaving to: {(EXPORT_ROOT / DATE_FOLDER).as_posix()}")
